In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP26A - CELL 1
# Environment + paths + load FROZEN Step25 objects
# ============================================================

from pathlib import Path

import sys
import json
import warnings

import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.stats import mannwhitneyu, spearmanr

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# Paths
# ============================================================

STEP23_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

STEP24_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step24_Bulk_to_scRNA_Projection')
)

STEP25_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step25_CrossPatient_Robustness')
)

STEP26_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step26_MS3_Biology')
)


DATA_DIR = STEP26_ROOT / "Data"
RESULTS_DIR = STEP26_ROOT / "Results"
FIGURES_DIR = STEP26_ROOT / "Figures"
LOGS_DIR = STEP26_ROOT / "Logs"
REFERENCE_DIR = STEP26_ROOT / "Reference"
NOTEBOOK_DIR = STEP26_ROOT / "Notebook"


for folder in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    REFERENCE_DIR,
    NOTEBOOK_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Frozen inputs
# ============================================================

STEP25_OBJECT = (
    STEP25_ROOT
    / "Data"
    / "GSE149614_STEP25_FINAL_CROSSPATIENT_ROBUST_MS3.h5ad"
)

CORE50_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core50.csv"
)

CORE100_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core100.csv"
)

STEP24_PB_LOG = (
    STEP24_ROOT
    / "Data"
    / "GSE149614_STEP24A_FinalState_Pseudobulk_LogCPM.h5ad"
)

STEP25_DECISION = (
    STEP25_ROOT
    / "Logs"
    / "Step25_FINAL_Decision.json"
)


for path in [
    STEP25_OBJECT,
    CORE50_INPUT,
    CORE100_INPUT,
    STEP24_PB_LOG,
    STEP25_DECISION
]:
    assert path.exists(), f"Missing file:\n{path}"


# ============================================================
# Load
# ============================================================

adata = sc.read_h5ad(
    STEP25_OBJECT
)

adata_pb_log = sc.read_h5ad(
    STEP24_PB_LOG
)

core50_table = pd.read_csv(
    CORE50_INPUT,
    index_col=0
)

core100_table = pd.read_csv(
    CORE100_INPUT,
    index_col=0
)


STATE_KEY = "final_malignant_state_id"

STATE_ORDER = [
    f"MS{i}"
    for i in range(1, 9)
]

PRIMARY_TARGET = "MS3"
PRIMARY_REFERENCE = "MS6"

MS3_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

MS6_PATIENTS = [
    "HCC09",
    "HCC10"
]

COMMON_AXIS_PATIENTS = [
    "HCC09",
    "HCC10"
]

MIN_CELLS_PER_PATIENT_STATE = 20


# ============================================================
# Recover frozen gene lists robustly
# ============================================================

def extract_gene_column(df):

    for col in [
        "Gene",
        "gene",
        "names"
    ]:
        if col in df.columns:
            return (
                df[col]
                .astype(str)
                .tolist()
            )

    # Fallback to index
    return (
        df.index
        .astype(str)
        .tolist()
    )


MS3_CORE50 = extract_gene_column(
    core50_table
)

MS3_CORE100 = extract_gene_column(
    core100_table
)


print("=" * 120)
print("STEP26A DEEP MOLECULAR CHARACTERIZATION OF MS3")
print("=" * 120)

print("\nPython:")
print(sys.executable)

print("\nScanpy:")
print(sc.__version__)

print("\nSingle-cell object:")
print(adata.shape)

print("\nPseudobulk object:")
print(adata_pb_log.shape)

print("\nFrozen MS3 Core50 genes:")
print(len(MS3_CORE50))

print("\nFrozen MS3 Core100 genes:")
print(len(MS3_CORE100))

print("\nImportant:")
print(
    "MS1-MS8 labels, Step24 bulk-risk roles and Step25 Core50/Core100 "
    "are FROZEN and will not be redefined."
)

In [ ]:
# ============================================================
# STEP26A - CELL 2
# Publication plotting settings
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False


mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename
):

    png_path = (
        FIGURES_DIR /
        f"{filename}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{filename}.pdf"
    )

    # Save before show
    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print("[OK]", png_path.name)
    print("[OK]", pdf_path.name)

In [ ]:
# ============================================================
# STEP26A - CELL 3
# MS3 / MS6 cohort audit
# ============================================================

patient_state_counts = pd.crosstab(

    adata.obs[
        "patient"
    ].astype(str),

    adata.obs[
        STATE_KEY
    ].astype(str)
)


patient_state_counts = (
    patient_state_counts
    .reindex(
        columns=STATE_ORDER,
        fill_value=0
    )
)


display(
    patient_state_counts
)


audit_rows = []


for patient in sorted(
    adata.obs[
        "patient"
    ]
    .astype(str)
    .unique()
):

    for state in [
        PRIMARY_TARGET,
        PRIMARY_REFERENCE
    ]:

        n = int(
            patient_state_counts.loc[
                patient,
                state
            ]
            if patient in patient_state_counts.index
            else 0
        )

        audit_rows.append({

            "Patient":
                patient,

            "State":
                state,

            "N_cells":
                n,

            "Eligible":
                n >= MIN_CELLS_PER_PATIENT_STATE
        })


ms3_ms6_audit = pd.DataFrame(
    audit_rows
)


AUDIT_PATH = (
    RESULTS_DIR /
    "Step26A_MS3_MS6_CellCount_Audit.csv"
)


ms3_ms6_audit.to_csv(
    AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    ms3_ms6_audit
)

In [ ]:
# ============================================================
# STEP26A - CELL 4
# Patient-aware MS3 vs other malignant-state gene delta
# ============================================================

pb_obs = (
    adata_pb_log.obs
    .copy()
)


pb_obs[
    "patient"
] = (
    pb_obs[
        "patient"
    ]
    .astype(str)
)


pb_obs[
    "Final_state_id"
] = (
    pb_obs[
        "Final_state_id"
    ]
    .astype(str)
)


X_pb = (
    adata_pb_log.X
)


if sparse.issparse(
    X_pb
):

    X_pb = (
        X_pb
        .toarray()
        .astype(np.float32)
    )

else:

    X_pb = np.asarray(
        X_pb,
        dtype=np.float32
    )


gene_names = (
    adata_pb_log.var_names
    .astype(str)
    .tolist()
)


ms3_delta_vectors = {}
ms3_delta_audit_rows = []


for patient in MS3_PATIENTS:

    target_mask = (

        (
            pb_obs[
                "patient"
            ]
            ==
            patient
        )

        &

        (
            pb_obs[
                "Final_state_id"
            ]
            ==
            PRIMARY_TARGET
        )

        &

        (
            pb_obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    target_idx = np.where(
        target_mask.values
    )[0]


    assert len(target_idx) == 1


    comparator_mask = (

        (
            pb_obs[
                "patient"
            ]
            ==
            patient
        )

        &

        (
            pb_obs[
                "Final_state_id"
            ]
            !=
            PRIMARY_TARGET
        )

        &

        (
            pb_obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    comparator_idx = np.where(
        comparator_mask.values
    )[0]


    assert len(comparator_idx) >= 1


    target_expression = (
        X_pb[
            target_idx[0],
            :
        ]
    )


    comparator_mean = np.mean(

        X_pb[
            comparator_idx,
            :
        ],

        axis=0
    )


    delta = (
        target_expression
        -
        comparator_mean
    )


    ms3_delta_vectors[
        patient
    ] = delta


    ms3_delta_audit_rows.append({

        "Patient":
            patient,

        "MS3_cells":
            int(
                pb_obs.iloc[
                    target_idx[0]
                ][
                    "N_cells"
                ]
            ),

        "N_comparator_states":
            len(
                comparator_idx
            ),

        "Comparator_states":
            "|".join(

                pb_obs.iloc[
                    comparator_idx
                ][
                    "Final_state_id"
                ]
                .astype(str)
                .tolist()
            )
    })


ms3_gene_delta = pd.DataFrame(

    ms3_delta_vectors,

    index=
        gene_names
)


ms3_gene_delta.index.name = (
    "Gene"
)


MS3_DELTA_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_vs_OtherStates_PatientAware_GeneDelta.csv"
)


ms3_gene_delta.to_csv(
    MS3_DELTA_PATH,
    encoding="utf-8-sig"
)


display(
    pd.DataFrame(
        ms3_delta_audit_rows
    )
)

In [ ]:
# ============================================================
# STEP26A - CELL 5
# Patient-consistency gene master table
# ============================================================

ms3_gene_master = pd.DataFrame(
    index=
        ms3_gene_delta.index
)


for patient in MS3_PATIENTS:

    ms3_gene_master[
        f"Delta_{patient}"
    ] = (
        ms3_gene_delta[
            patient
        ]
    )


delta_cols = [

    f"Delta_{patient}"
    for patient in MS3_PATIENTS
]


ms3_gene_master[
    "Mean_delta"
] = (
    ms3_gene_master[
        delta_cols
    ]
    .mean(
        axis=1
    )
)


ms3_gene_master[
    "Median_delta"
] = (
    ms3_gene_master[
        delta_cols
    ]
    .median(
        axis=1
    )
)


ms3_gene_master[
    "Min_delta"
] = (
    ms3_gene_master[
        delta_cols
    ]
    .min(
        axis=1
    )
)


ms3_gene_master[
    "Max_delta"
] = (
    ms3_gene_master[
        delta_cols
    ]
    .max(
        axis=1
    )
)


ms3_gene_master[
    "Positive_fraction"
] = (

    (
        ms3_gene_master[
            delta_cols
        ]
        >
        0
    )
    .mean(
        axis=1
    )
)


ms3_gene_master[
    "Negative_fraction"
] = (

    (
        ms3_gene_master[
            delta_cols
        ]
        <
        0
    )
    .mean(
        axis=1
    )
)


ms3_gene_master[
    "Direction"
] = np.select(

    [
        ms3_gene_master[
            "Positive_fraction"
        ]
        ==
        1.0,

        ms3_gene_master[
            "Negative_fraction"
        ]
        ==
        1.0
    ],

    [
        "Up_in_all_MS3_patients",
        "Down_in_all_MS3_patients"
    ],

    default=
        "Mixed_direction"
)


ms3_gene_master[
    "Abs_mean_delta"
] = (
    ms3_gene_master[
        "Mean_delta"
    ]
    .abs()
)


ms3_gene_master[
    "Is_Core50"
] = (
    ms3_gene_master.index
    .isin(
        MS3_CORE50
    )
)


ms3_gene_master[
    "Is_Core100"
] = (
    ms3_gene_master.index
    .isin(
        MS3_CORE100
    )
)


ms3_gene_master[
    "Is_MT"
] = (
    ms3_gene_master.index
    .str.upper()
    .str.startswith(
        "MT-"
    )
)


ms3_gene_master[
    "Is_Ribosomal"
] = (
    ms3_gene_master.index
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
)


MS3_MASTER_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_PatientConsistent_GeneMaster.csv"
)


ms3_gene_master.to_csv(
    MS3_MASTER_PATH,
    encoding="utf-8-sig"
)


print("=" * 110)
print("MS3 PATIENT-CONSISTENT GENE MASTER")
print("=" * 110)

print(
    ms3_gene_master[
        "Direction"
    ]
    .value_counts()
)


print("\nTop consistent upregulated genes:")

display(

    ms3_gene_master[
        ms3_gene_master[
            "Direction"
        ]
        ==
        "Up_in_all_MS3_patients"
    ]

    .sort_values(
        "Min_delta",
        ascending=False
    )

    .head(40)

    .round(4)
)


print("\nTop consistent downregulated genes:")

display(

    ms3_gene_master[
        ms3_gene_master[
            "Direction"
        ]
        ==
        "Down_in_all_MS3_patients"
    ]

    .sort_values(
        "Max_delta",
        ascending=True
    )

    .head(40)

    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 6
# Paired MS3 vs MS6 pseudobulk contrast
# HCC09 / HCC10 only
# ============================================================

axis_vectors = {}
axis_audit_rows = []


for patient in COMMON_AXIS_PATIENTS:

    ms3_mask = (

        (
            pb_obs[
                "patient"
            ]
            ==
            patient
        )

        &

        (
            pb_obs[
                "Final_state_id"
            ]
            ==
            PRIMARY_TARGET
        )

        &

        (
            pb_obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    ms6_mask = (

        (
            pb_obs[
                "patient"
            ]
            ==
            patient
        )

        &

        (
            pb_obs[
                "Final_state_id"
            ]
            ==
            PRIMARY_REFERENCE
        )

        &

        (
            pb_obs[
                "N_cells"
            ]
            >=
            MIN_CELLS_PER_PATIENT_STATE
        )
    )


    ms3_idx = np.where(
        ms3_mask.values
    )[0]

    ms6_idx = np.where(
        ms6_mask.values
    )[0]


    assert len(ms3_idx) == 1
    assert len(ms6_idx) == 1


    delta = (

        X_pb[
            ms3_idx[0],
            :
        ]

        -

        X_pb[
            ms6_idx[0],
            :
        ]
    )


    axis_vectors[
        patient
    ] = delta


    axis_audit_rows.append({

        "Patient":
            patient,

        "MS3_cells":
            int(
                pb_obs.iloc[
                    ms3_idx[0]
                ][
                    "N_cells"
                ]
            ),

        "MS6_cells":
            int(
                pb_obs.iloc[
                    ms6_idx[0]
                ][
                    "N_cells"
                ]
            )
    })


ms3_vs_ms6_gene_delta = pd.DataFrame(

    axis_vectors,

    index=
        gene_names
)


ms3_vs_ms6_gene_delta.index.name = (
    "Gene"
)


axis_gene_master = pd.DataFrame(
    index=
        gene_names
)


for patient in COMMON_AXIS_PATIENTS:

    axis_gene_master[
        f"Delta_{patient}"
    ] = (
        ms3_vs_ms6_gene_delta[
            patient
        ]
    )


axis_delta_cols = [

    f"Delta_{patient}"
    for patient in COMMON_AXIS_PATIENTS
]


axis_gene_master[
    "Mean_delta"
] = (
    axis_gene_master[
        axis_delta_cols
    ]
    .mean(
        axis=1
    )
)


axis_gene_master[
    "Min_delta"
] = (
    axis_gene_master[
        axis_delta_cols
    ]
    .min(
        axis=1
    )
)


axis_gene_master[
    "Max_delta"
] = (
    axis_gene_master[
        axis_delta_cols
    ]
    .max(
        axis=1
    )
)


axis_gene_master[
    "Direction"
] = np.select(

    [
        (
            axis_gene_master[
                axis_delta_cols
            ]
            >
            0
        )
        .all(
            axis=1
        ),

        (
            axis_gene_master[
                axis_delta_cols
            ]
            <
            0
        )
        .all(
            axis=1
        )
    ],

    [
        "MS3_higher_in_both_patients",
        "MS6_higher_in_both_patients"
    ],

    default=
        "Mixed_direction"
)


AXIS_GENE_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_vs_MS6_Paired_GeneDelta.csv"
)


axis_gene_master.to_csv(
    AXIS_GENE_PATH,
    encoding="utf-8-sig"
)


display(
    pd.DataFrame(
        axis_audit_rows
    )
)


print("\nTop MS3-higher genes:")

display(

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS3_higher_in_both_patients"
    ]

    .sort_values(
        "Min_delta",
        ascending=False
    )

    .head(30)

    .round(4)
)


print("\nTop MS6-higher genes:")

display(

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS6_higher_in_both_patients"
    ]

    .sort_values(
        "Max_delta",
        ascending=True
    )

    .head(30)

    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 7
# Curated deep biological programs
# ============================================================

DEEP_PROGRAMS = {

    # --------------------------------------------------------
    # Cell-cycle architecture
    # --------------------------------------------------------

    "Mitotic_Entry": [
        "CDK1", "CCNB1", "CCNB2", "CDC25C",
        "PLK1", "AURKA", "AURKB", "BORA"
    ],

    "Spindle_Chromosome_Segregation": [
        "CENPA", "CENPE", "CENPF", "KIF2C",
        "KIF11", "KIF20A", "BUB1", "BUB1B",
        "MAD2L1", "NDC80", "NUF2", "SPC25"
    ],

    "Cytokinesis": [
        "PRC1", "KIF23", "KIF20B", "ECT2",
        "RACGAP1", "ANLN", "CEP55", "AURKB"
    ],

    # --------------------------------------------------------
    # DNA replication / replication stress
    # --------------------------------------------------------

    "DNA_Replication_Origin": [
        "MCM2", "MCM3", "MCM4", "MCM5",
        "MCM6", "MCM7", "CDC6", "CDT1",
        "ORC1", "GINS1", "GINS2", "GINS3"
    ],

    "Replication_Stress": [
        "ATR", "CHEK1", "CLSPN", "TIMELESS",
        "TIPIN", "RRM2", "WEE1", "CDC45",
        "MCM10", "RAD18"
    ],

    # --------------------------------------------------------
    # DNA repair
    # --------------------------------------------------------

    "Homologous_Recombination": [
        "BRCA1", "BRCA2", "RAD51", "RAD51AP1",
        "EXO1", "BLM", "RMI1", "MRE11",
        "NBN", "PALB2"
    ],

    "Mismatch_Repair": [
        "MSH2", "MSH6", "MLH1", "PMS2",
        "EXO1", "RFC1", "RFC4"
    ],

    "Fanconi_DNA_Repair": [
        "FANCD2", "FANCI", "FANCA", "FANCC",
        "FANCG", "BRIP1", "UBE2T"
    ],

    # --------------------------------------------------------
    # Chromosomal instability
    # --------------------------------------------------------

    "Chromosomal_Instability": [
        "AURKA", "AURKB", "PLK1", "BUB1",
        "BUB1B", "MAD2L1", "TPX2", "NEK2",
        "CENPE", "CENPF", "KIF2C"
    ],

    # --------------------------------------------------------
    # Nucleotide / anabolic metabolism
    # --------------------------------------------------------

    "Nucleotide_Synthesis": [
        "RRM1", "RRM2", "TYMS", "DHFR",
        "CAD", "GMPS", "IMPDH1", "IMPDH2",
        "PPAT", "PAICS", "ATIC"
    ],

    "OneCarbon_Metabolism": [
        "SHMT1", "SHMT2", "MTHFD1", "MTHFD2",
        "MTHFD1L", "PHGDH", "PSAT1", "PSPH"
    ],

    # --------------------------------------------------------
    # Energy metabolism
    # --------------------------------------------------------

    "Glycolysis_Core": [
        "HK2", "GPI", "PFKP", "ALDOA",
        "GAPDH", "PGK1", "ENO1", "PKM",
        "LDHA", "SLC2A1"
    ],

    "Oxidative_Phosphorylation_Nuclear": [
        "NDUFS1", "NDUFS2", "NDUFS3",
        "SDHA", "SDHB",
        "UQCRC1", "UQCRC2",
        "COX4I1", "COX5A",
        "ATP5F1A", "ATP5F1B"
    ],

    # --------------------------------------------------------
    # Proteostasis / stress
    # --------------------------------------------------------

    "UPR_Proteostasis": [
        "HSPA5", "HSP90B1", "XBP1", "ATF4",
        "DDIT3", "DNAJB9", "HERPUD1",
        "CALR", "CANX"
    ],

    "Oxidative_Stress": [
        "NFE2L2", "KEAP1", "NQO1", "HMOX1",
        "GCLC", "GCLM", "TXNRD1", "SOD1",
        "SOD2", "GPX4"
    ],

    # --------------------------------------------------------
    # Cell-death / survival
    # --------------------------------------------------------

    "Apoptosis_Resistance": [
        "BCL2", "BCL2L1", "MCL1",
        "BIRC5", "XIAP", "BIRC2",
        "BIRC3"
    ],

    # --------------------------------------------------------
    # Differentiation / metabolic reference
    # --------------------------------------------------------

    "Hepatic_Differentiation": [
        "ALB", "APOA1", "APOA2", "TTR",
        "ASGR1", "ASGR2", "CPS1", "HPD",
        "MAT1A", "HNF4A", "PCK1", "F7"
    ],

    "Bile_Acid_Metabolism": [
        "CYP8B1", "CYP7A1", "SLC10A1",
        "ABCB11", "ABCC2", "BAAT",
        "SULT2A1", "NR1H4", "AKR1D1"
    ],

    "Fatty_Acid_Oxidation": [
        "CPT1A", "CPT2", "ACADM", "ACADL",
        "HADHA", "HADHB", "ACAA2",
        "ECHS1", "PPARA", "HMGCS2"
    ]
}


program_audit_rows = []


for program, genes in DEEP_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata.var_names
    ]

    missing = [
        gene
        for gene in genes
        if gene not in adata.var_names
    ]

    program_audit_rows.append({

        "Program":
            program,

        "N_defined":
            len(genes),

        "N_detected":
            len(detected),

        "Detected_fraction":
            len(detected) / len(genes),

        "Detected_genes":
            "|".join(detected),

        "Missing_genes":
            "|".join(missing)
    })


deep_program_audit = pd.DataFrame(
    program_audit_rows
)


DEEP_PROGRAM_AUDIT_PATH = (

    RESULTS_DIR /
    "Step26A_DeepProgram_GeneAudit.csv"
)


deep_program_audit.to_csv(

    DEEP_PROGRAM_AUDIT_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    deep_program_audit[
        [
            "Program",
            "N_defined",
            "N_detected",
            "Detected_fraction"
        ]
    ]
    .round(3)
)

In [ ]:
# ============================================================
# STEP26A - CELL 8
# Score deep biological programs
# ============================================================

adata26 = (
    adata.copy()
)


deep_score_cols = []


for program, genes in DEEP_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata26.var_names
    ]


    if len(detected) < 3:

        print(
            f"[SKIP] {program}: only {len(detected)} detected genes"
        )

        continue


    score_col = (
        "DeepProgram__"
        +
        program
    )


    sc.tl.score_genes(

        adata26,

        gene_list=
            detected,

        score_name=
            score_col,

        ctrl_size=
            min(
                50,
                max(
                    15,
                    len(detected) * 3
                )
            ),

        n_bins=
            25,

        random_state=
            SEED,

        use_raw=
            False
    )


    deep_score_cols.append(
        score_col
    )


print(
    "Deep biological programs scored:",
    len(
        deep_score_cols
    )
)

In [ ]:
# ============================================================
# STEP26A - CELL 8
# Score deep biological programs
# ============================================================

adata26 = (
    adata.copy()
)


deep_score_cols = []


for program, genes in DEEP_PROGRAMS.items():

    detected = [
        gene
        for gene in genes
        if gene in adata26.var_names
    ]


    if len(detected) < 3:

        print(
            f"[SKIP] {program}: only {len(detected)} detected genes"
        )

        continue


    score_col = (
        "DeepProgram__"
        +
        program
    )


    sc.tl.score_genes(

        adata26,

        gene_list=
            detected,

        score_name=
            score_col,

        ctrl_size=
            min(
                50,
                max(
                    15,
                    len(detected) * 3
                )
            ),

        n_bins=
            25,

        random_state=
            SEED,

        use_raw=
            False
    )


    deep_score_cols.append(
        score_col
    )


print(
    "Deep biological programs scored:",
    len(
        deep_score_cols
    )
)

In [ ]:
# ============================================================
# STEP26A - CELL 9
# Patient × state deep-program medians
# ============================================================

program_df = (

    adata26.obs[
        [
            "patient",
            STATE_KEY
        ]
        +
        deep_score_cols
    ]
    .copy()
)


program_df[
    "patient"
] = (
    program_df[
        "patient"
    ]
    .astype(str)
)


program_df[
    STATE_KEY
] = (
    program_df[
        STATE_KEY
    ]
    .astype(str)
)


patient_state_program = (

    program_df

    .groupby(
        [
            "patient",
            STATE_KEY
        ],
        observed=True
    )[
        deep_score_cols
    ]

    .median()

    .reset_index()
)


cell_count_df = (

    adata26.obs

    .assign(

        patient_string=
            adata26.obs[
                "patient"
            ]
            .astype(str),

        state_string=
            adata26.obs[
                STATE_KEY
            ]
            .astype(str)
    )

    .groupby(
        [
            "patient_string",
            "state_string"
        ]
    )

    .size()

    .rename(
        "N_cells"
    )

    .reset_index()
)


cell_count_df.columns = [

    "patient",
    STATE_KEY,
    "N_cells"
]


patient_state_program = (

    patient_state_program

    .merge(

        cell_count_df,

        on=[
            "patient",
            STATE_KEY
        ],

        how="left",

        validate=
            "one_to_one"
    )
)


display(
    patient_state_program
    .head(20)
)

In [ ]:
# ============================================================
# STEP26A - CELL 10
# Patient-aware deep-program delta for MS3
# ============================================================

ms3_program_rows = []


for patient in MS3_PATIENTS:

    target = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                ==
                PRIMARY_TARGET
            )

            &

            (
                patient_state_program[
                    "N_cells"
                ]
                >=
                MIN_CELLS_PER_PATIENT_STATE
            )
        ]
    )


    assert len(target) == 1


    comparators = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                !=
                PRIMARY_TARGET
            )

            &

            (
                patient_state_program[
                    "N_cells"
                ]
                >=
                MIN_CELLS_PER_PATIENT_STATE
            )
        ]
    )


    assert len(comparators) >= 1


    out = {

        "Patient":
            patient,

        "MS3_cells":
            int(
                target.iloc[0][
                    "N_cells"
                ]
            ),

        "N_comparator_states":
            len(
                comparators
            )
    }


    for col in deep_score_cols:

        program_name = (
            col.replace(
                "DeepProgram__",
                ""
            )
        )


        delta = (

            float(
                target.iloc[0][col]
            )

            -

            float(
                comparators[col]
                .mean()
            )
        )


        out[
            program_name
        ] = delta


    ms3_program_rows.append(
        out
    )


ms3_deep_program_delta = pd.DataFrame(
    ms3_program_rows
)


MS3_PROGRAM_DELTA_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_PatientAware_DeepProgramDelta.csv"
)


ms3_deep_program_delta.to_csv(

    MS3_PROGRAM_DELTA_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    ms3_deep_program_delta
    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 11
# Cross-patient conservation of deep MS3 programs
# ============================================================

meta_cols = {
    "Patient",
    "MS3_cells",
    "N_comparator_states"
}


deep_program_names = [

    col
    for col in ms3_deep_program_delta.columns
    if col not in meta_cols
]


program_conservation_rows = []


for program in deep_program_names:

    values = (

        ms3_deep_program_delta[
            program
        ]
        .astype(float)
    )


    program_conservation_rows.append({

        "Program":
            program,

        "N_patients":
            len(values),

        "Mean_delta":
            float(
                values.mean()
            ),

        "Median_delta":
            float(
                values.median()
            ),

        "Min_delta":
            float(
                values.min()
            ),

        "Max_delta":
            float(
                values.max()
            ),

        "Positive_fraction":
            float(
                np.mean(
                    values > 0
                )
            ),

        "Negative_fraction":
            float(
                np.mean(
                    values < 0
                )
            ),

        "Direction_consensus":
            (
                "Positive_all"
                if np.all(values > 0)

                else

                "Negative_all"
                if np.all(values < 0)

                else

                "Mixed"
            )
    })


deep_program_conservation = (

    pd.DataFrame(
        program_conservation_rows
    )

    .sort_values(

        [
            "Positive_fraction",
            "Mean_delta"
        ],

        ascending=[
            False,
            False
        ]
    )
)


DEEP_PROGRAM_CONSERVATION_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_DeepProgram_Conservation.csv"
)


deep_program_conservation.to_csv(

    DEEP_PROGRAM_CONSERVATION_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    deep_program_conservation
    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 12
# Paired MS3 vs MS6 deep-program contrast
# ============================================================

axis_program_rows = []


for patient in COMMON_AXIS_PATIENTS:

    ms3_row = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                ==
                PRIMARY_TARGET
            )
        ]
        .iloc[0]
    )


    ms6_row = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                ==
                PRIMARY_REFERENCE
            )
        ]
        .iloc[0]
    )


    out = {

        "Patient":
            patient
    }


    for col in deep_score_cols:

        program_name = (
            col.replace(
                "DeepProgram__",
                ""
            )
        )


        out[
            program_name
        ] = (

            float(
                ms3_row[col]
            )

            -

            float(
                ms6_row[col]
            )
        )


    axis_program_rows.append(
        out
    )


ms3_vs_ms6_program_delta = pd.DataFrame(
    axis_program_rows
)


AXIS_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_vs_MS6_DeepProgramDelta.csv"
)


ms3_vs_ms6_program_delta.to_csv(

    AXIS_PROGRAM_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    ms3_vs_ms6_program_delta
    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 13
# Cross-patient deep-program heatmap
# ============================================================

heat = (

    ms3_deep_program_delta

    .set_index(
        "Patient"
    )[
        deep_program_names
    ]

    .T
)


# Order by mean MS3 delta
program_order = (

    heat.mean(
        axis=1
    )

    .sort_values(
        ascending=False
    )

    .index
)


heat = (
    heat.loc[
        program_order
    ]
)


fig, ax = plt.subplots(
    figsize=(7.5, 9.5)
)


values = heat.values


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        heat.shape[1]
    )
)


ax.set_xticklabels(
    heat.columns
)


ax.set_yticks(
    np.arange(
        heat.shape[0]
    )
)


ax.set_yticklabels(

    [
        x.replace(
            "_",
            " "
        )

        for x in heat.index
    ]
)


ax.set_xlabel(
    "Patient"
)

ax.set_ylabel(
    "Biological Program"
)

ax.set_title(
    "Cross-patient Molecular Architecture of MS3"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.025
)


cbar.set_label(
    "Within-patient Program Delta"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step26A_01_MS3_DeepProgram_CrossPatientHeatmap"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP26A - CELL 14
# MS3 vs MS6 paired-axis heatmap
# ============================================================

axis_heat = (

    ms3_vs_ms6_program_delta

    .set_index(
        "Patient"
    )[
        deep_program_names
    ]

    .T
)


axis_order = (

    axis_heat.mean(
        axis=1
    )

    .sort_values(
        ascending=False
    )

    .index
)


axis_heat = (
    axis_heat.loc[
        axis_order
    ]
)


fig, ax = plt.subplots(
    figsize=(6.6, 9.5)
)


values = (
    axis_heat.values
)


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        axis_heat.shape[1]
    )
)


ax.set_xticklabels(
    axis_heat.columns
)


ax.set_yticks(
    np.arange(
        axis_heat.shape[0]
    )
)


ax.set_yticklabels(

    [
        x.replace(
            "_",
            " "
        )

        for x in axis_heat.index
    ]
)


ax.set_xlabel(
    "Patient"
)

ax.set_ylabel(
    "Biological Program"
)

ax.set_title(
    "MS3 versus MS6 Molecular-state Axis"
)


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.04,
    pad=0.03
)


cbar.set_label(
    "Program Difference (MS3 - MS6)"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step26A_02_MS3_vs_MS6_ProgramAxis"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP26A - CELL 15
# Functional architecture of frozen MS3 Core50
# ============================================================

gene_to_programs = {}


for program, genes in DEEP_PROGRAMS.items():

    for gene in genes:

        gene_to_programs.setdefault(
            gene,
            []
        )

        gene_to_programs[
            gene
        ].append(
            program
        )


core50_function_rows = []


for rank, gene in enumerate(
    MS3_CORE50,
    start=1
):

    programs = (
        gene_to_programs.get(
            gene,
            []
        )
    )


    core50_function_rows.append({

        "Rank":
            rank,

        "Gene":
            gene,

        "N_curated_programs":
            len(programs),

        "Curated_programs":
            "|".join(
                programs
            )
            if len(programs) > 0
            else
            "Unassigned_in_curated_programs"
    })


core50_function = pd.DataFrame(
    core50_function_rows
)


CORE50_FUNCTION_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_Core50_FunctionalArchitecture.csv"
)


core50_function.to_csv(

    CORE50_FUNCTION_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    core50_function
)

In [ ]:
# ============================================================
# STEP26A - CELL 16
# Residual conserved MS3 genes beyond frozen Core100
# ============================================================

residual_ms3 = (

    ms3_gene_master[
        (
            ms3_gene_master[
                "Direction"
            ]
            ==
            "Up_in_all_MS3_patients"
        )

        &

        (
            ~ms3_gene_master[
                "Is_Core100"
            ]
        )

        &

        (
            ~ms3_gene_master[
                "Is_MT"
            ]
        )

        &

        (
            ~ms3_gene_master[
                "Is_Ribosomal"
            ]
        )
    ]

    .sort_values(

        [
            "Min_delta",
            "Mean_delta"
        ],

        ascending=[
            False,
            False
        ]
    )

    .copy()
)


RESIDUAL_MS3_PATH = (

    RESULTS_DIR /
    "Step26A_MS3_ResidualConservedGenes_BeyondCore100.csv"
)


residual_ms3.to_csv(
    RESIDUAL_MS3_PATH,
    encoding="utf-8-sig"
)


print(
    "Residual conserved MS3 genes beyond Core100:",
    len(
        residual_ms3
    )
)


display(
    residual_ms3
    .head(50)
    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 17
# OPTIONAL offline GMT enrichment
# ============================================================

from scipy.stats import hypergeom


GMT_DIR = (
    REFERENCE_DIR /
    "GMT"
)

GMT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


gmt_files = list(
    GMT_DIR.glob(
        "*.gmt"
    )
)


def parse_gmt(
    path
):

    gene_sets = {}


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            parts = (
                line.rstrip("\n")
                .split("\t")
            )


            if len(parts) < 3:
                continue


            pathway = (
                parts[0]
            )

            genes = set(
                parts[2:]
            )


            gene_sets[
                pathway
            ] = genes


    return gene_sets


def bh_adjust(
    pvalues
):

    pvalues = np.asarray(
        pvalues,
        dtype=float
    )

    n = len(pvalues)

    if n == 0:
        return pvalues


    order = np.argsort(
        pvalues
    )

    ranked = (
        pvalues[
            order
        ]
    )


    adjusted = (

        ranked
        *
        n
        /
        np.arange(
            1,
            n + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    output = np.empty(
        n
    )

    output[
        order
    ] = adjusted


    return output


# ============================================================
# Query sets
# ============================================================

query_sets = {

    "MS3_Conserved_Top300":
        set(

            ms3_gene_master[
                ms3_gene_master[
                    "Direction"
                ]
                ==
                "Up_in_all_MS3_patients"
            ]

            .sort_values(
                "Min_delta",
                ascending=False
            )

            .head(300)

            .index
        ),

    "MS3_Residual_BeyondCore100_Top300":
        set(

            residual_ms3
            .head(300)
            .index
        ),

    "MS6_Direction_Top300":
        set(

            axis_gene_master[
                axis_gene_master[
                    "Direction"
                ]
                ==
                "MS6_higher_in_both_patients"
            ]

            .sort_values(
                "Max_delta",
                ascending=True
            )

            .head(300)

            .index
        )
}


ora_results = []


if len(gmt_files) == 0:

    print(
        "No local GMT files found."
    )

    print(
        "Offline enrichment skipped."
    )


else:

    background = set(
        adata.var_names
    )

    M = len(
        background
    )


    for gmt_path in gmt_files:

        print(
            "Reading:",
            gmt_path.name
        )


        gene_sets = parse_gmt(
            gmt_path
        )


        for query_name, query_genes in (
            query_sets.items()
        ):

            query = (
                query_genes
                &
                background
            )

            N = len(
                query
            )


            if N == 0:
                continue


            local_rows = []


            for pathway, genes in (
                gene_sets.items()
            ):

                genes = (
                    genes
                    &
                    background
                )

                K = len(
                    genes
                )


                overlap = (
                    query
                    &
                    genes
                )


                k = len(
                    overlap
                )


                if k < 3:
                    continue


                p = hypergeom.sf(
                    k - 1,
                    M,
                    K,
                    N
                )


                local_rows.append({

                    "GMT_file":
                        gmt_path.name,

                    "Query":
                        query_name,

                    "Pathway":
                        pathway,

                    "Query_size":
                        N,

                    "Pathway_size":
                        K,

                    "Overlap":
                        k,

                    "P_value":
                        p,

                    "Genes":
                        "|".join(
                            sorted(
                                overlap
                            )
                        )
                })


            if len(local_rows) > 0:

                local_df = pd.DataFrame(
                    local_rows
                )


                local_df[
                    "FDR_BH"
                ] = bh_adjust(
                    local_df[
                        "P_value"
                    ].values
                )


                ora_results.append(
                    local_df
                )


if len(ora_results) > 0:

    ora_df = pd.concat(
        ora_results,
        ignore_index=True
    )


    ora_df = (

        ora_df

        .sort_values(
            [
                "Query",
                "FDR_BH",
                "P_value"
            ]
        )
    )


else:

    ora_df = pd.DataFrame()


ORA_PATH = (

    RESULTS_DIR /
    "Step26A_Offline_GMT_Enrichment.csv"
)


ora_df.to_csv(

    ORA_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "GMT files:",
    len(
        gmt_files
    )
)

print(
    "ORA rows:",
    len(
        ora_df
    )
)

In [ ]:
# ============================================================
# STEP26A - CELL 18
# Master molecular evidence table
# ============================================================

master_program = (
    deep_program_conservation
    .copy()
)


axis_program_mean = (

    ms3_vs_ms6_program_delta

    .set_index(
        "Patient"
    )[
        deep_program_names
    ]

    .mean(
        axis=0
    )

    .rename(
        "Mean_MS3_minus_MS6"
    )

    .reset_index()
)


axis_program_mean.columns = [
    "Program",
    "Mean_MS3_minus_MS6"
]


master_program = (

    master_program

    .merge(

        axis_program_mean,

        on=
            "Program",

        how=
            "left"
    )
)


MASTER_PROGRAM_PATH = (

    RESULTS_DIR /
    "Step26A_Master_MS3_MolecularProgram_Evidence.csv"
)


master_program.to_csv(

    MASTER_PROGRAM_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    master_program
    .round(4)
)

In [ ]:
# ============================================================
# STEP26A - CELL 19
# Save Step26A checkpoint
# ============================================================

adata26.uns[
    "Step26A_MS3_deep_biology"
] = {

    "Status":
        "EVIDENCE_CONSTRUCTION_COMPLETE",

    "Primary_state":
        "MS3",

    "Primary_reference":
        "MS6",

    "Frozen_primary_signature":
        "MS3_Conserved_Core50",

    "MS3_patients":
        MS3_PATIENTS,

    "MS3_MS6_common_patients":
        COMMON_AXIS_PATIENTS,

    "Deep_program_count":
        len(
            deep_score_cols
        ),

    "State_labels_modified":
        False,

    "Risk_roles_modified":
        False,

    "Next":
        (
            "Review gene-level and program-level molecular architecture "
            "before pathway enrichment and candidate vulnerability analysis."
        )
}


STEP26A_OBJECT = (

    DATA_DIR /
    "GSE149614_STEP26A_MS3_DEEP_BIOLOGY_CHECKPOINT.h5ad"
)


adata26.write_h5ad(

    STEP26A_OBJECT,

    compression="gzip"
)


print(
    "Saved:",
    STEP26A_OBJECT
)

In [ ]:
# ============================================================
# STEP26A - CELL 20
# Decision / methodology log
# ============================================================

step26a_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step26A_MS3_Deep_Molecular_Characterization",

    "Status":
        "COMPLETE_PENDING_REVIEW",

    "Primary_state":
        "MS3 Cycling-G2M",

    "Reference_state":
        "MS6 Hepatic-Differentiated",

    "MS3_supported_patients":
        MS3_PATIENTS,

    "MS3_MS6_common_patients":
        COMMON_AXIS_PATIENTS,

    "Frozen_signature":
        "MS3 Conserved Core50",

    "Primary_gene_contrast":
        (
            "MS3 pseudobulk minus the mean of the same patient's "
            "other eligible malignant-state pseudobulks."
        ),

    "Primary_axis_contrast":
        (
            "Paired MS3 minus MS6 pseudobulk in HCC09 and HCC10."
        ),

    "Deep_program_strategy":
        (
            "Transparent curated biological programs were scored "
            "for replication, DNA repair, chromosome segregation, "
            "metabolism, stress, survival and differentiation. "
            "These are not claimed to be original MSigDB gene sets."
        ),

    "Purpose":
        (
            "Identify conserved molecular architecture of MS3 beyond "
            "its already-established Cycling-G2M identity."
        ),

    "State_labels_modified":
        False,

    "Step24_risk_roles_modified":
        False,

    "Step25_Core50_modified":
        False,

    "Next_step":
        (
            "Review Step26A results; then perform pathway-level "
            "enrichment and prioritize mechanistically interpretable "
            "MS3 vulnerabilities in Step26B."
        )
}


SUMMARY_PATH = (

    LOGS_DIR /
    "Step26A_Final_Summary.json"
)


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step26a_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_PATH
)

In [ ]:
# ============================================================
# STEP26A - CELL 21
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR),
    ("Reference", REFERENCE_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (

    RESULTS_DIR /
    "Step26A_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP26A - CELL 22
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 125)
print("STEP26A FINAL MS3 DEEP-BIOLOGY REVIEW")
print("=" * 125)


print("\n[1] MS3 patient-aware gene architecture")
print("-" * 125)

print(
    ms3_gene_master[
        "Direction"
    ]
    .value_counts()
)


print("\nTop consistent MS3-up genes:")

display(

    ms3_gene_master[
        ms3_gene_master[
            "Direction"
        ]
        ==
        "Up_in_all_MS3_patients"
    ]

    .sort_values(
        "Min_delta",
        ascending=False
    )

    .head(30)

    .round(4)
)


print("\nTop consistent MS3-down genes:")

display(

    ms3_gene_master[
        ms3_gene_master[
            "Direction"
        ]
        ==
        "Down_in_all_MS3_patients"
    ]

    .sort_values(
        "Max_delta",
        ascending=True
    )

    .head(30)

    .round(4)
)


print("\n[2] MS3 vs MS6 paired gene axis")
print("-" * 125)

print(
    axis_gene_master[
        "Direction"
    ]
    .value_counts()
)


print("\nTop MS3-higher genes:")

display(

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS3_higher_in_both_patients"
    ]

    .sort_values(
        "Min_delta",
        ascending=False
    )

    .head(25)

    .round(4)
)


print("\nTop MS6-higher genes:")

display(

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS6_higher_in_both_patients"
    ]

    .sort_values(
        "Max_delta",
        ascending=True
    )

    .head(25)

    .round(4)
)


print("\n[3] MS3 deep-program conservation")
print("-" * 125)

display(
    deep_program_conservation
    .round(4)
)


print("\n[4] MS3 vs MS6 deep-program axis")
print("-" * 125)

display(
    ms3_vs_ms6_program_delta
    .round(4)
)


print("\n[5] Frozen Core50 functional architecture")
print("-" * 125)

display(
    core50_function
)


print("\n[6] Residual conserved genes beyond Core100")
print("-" * 125)

print(
    "N residual conserved genes:",
    len(
        residual_ms3
    )
)

display(
    residual_ms3
    .head(40)
    .round(4)
)


print("\n[7] Optional offline GMT enrichment")
print("-" * 125)

print(
    "GMT files detected:",
    len(
        gmt_files
    )
)

print(
    "ORA rows:",
    len(
        ora_df
    )
)


if len(
    ora_df
) > 0:

    display(
        ora_df
        .head(40)
    )


print("\n[8] Master molecular program evidence")
print("-" * 125)

display(
    master_program
    .round(4)
)


print("\n[9] Files")
print("-" * 125)


required_files = [

    STEP26A_OBJECT,
    AUDIT_PATH,
    MS3_DELTA_PATH,
    MS3_MASTER_PATH,
    AXIS_GENE_PATH,
    DEEP_PROGRAM_AUDIT_PATH,
    MS3_PROGRAM_DELTA_PATH,
    DEEP_PROGRAM_CONSERVATION_PATH,
    AXIS_PROGRAM_PATH,
    CORE50_FUNCTION_PATH,
    RESIDUAL_MS3_PATH,
    ORA_PATH,
    MASTER_PROGRAM_PATH,
    SUMMARY_PATH,
    MANIFEST_PATH

]


for path in required_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP26A STATUS = DEEP MOLECULAR EVIDENCE COMPLETE"
)

print(
    "MS3 STATE DEFINITION MODIFIED = NO"
)

print(
    "MS3 CORE50 MODIFIED = NO"
)

print(
    "NEXT = REVIEW RESULTS BEFORE STEP26B PATHWAY AND VULNERABILITY ANALYSIS"
)

print("=" * 125)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP26B - CELL 23
# Reload Step26A evidence + frozen Step25 signatures
# ============================================================

from pathlib import Path

import sys
import json
import warnings
import numpy as np
import pandas as pd

import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse
from scipy.stats import hypergeom

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# Paths
# ============================================================

STEP25_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step25_CrossPatient_Robustness')
)

STEP26_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step26_MS3_Biology')
)


DATA_DIR = STEP26_ROOT / "Data"
RESULTS_DIR = STEP26_ROOT / "Results"
FIGURES_DIR = STEP26_ROOT / "Figures"
LOGS_DIR = STEP26_ROOT / "Logs"
REFERENCE_DIR = STEP26_ROOT / "Reference"

GENESET_DIR = REFERENCE_DIR / "GeneSets"

for folder in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    REFERENCE_DIR,
    GENESET_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Inputs
# ============================================================

STEP26A_OBJECT = (
    DATA_DIR
    / "GSE149614_STEP26A_MS3_DEEP_BIOLOGY_CHECKPOINT.h5ad"
)

MS3_MASTER_INPUT = (
    RESULTS_DIR
    / "Step26A_MS3_PatientConsistent_GeneMaster.csv"
)

AXIS_GENE_INPUT = (
    RESULTS_DIR
    / "Step26A_MS3_vs_MS6_Paired_GeneDelta.csv"
)

RESIDUAL_INPUT = (
    RESULTS_DIR
    / "Step26A_MS3_ResidualConservedGenes_BeyondCore100.csv"
)

PROGRAM_INPUT = (
    RESULTS_DIR
    / "Step26A_Master_MS3_MolecularProgram_Evidence.csv"
)

CORE50_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core50.csv"
)

CORE100_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core100.csv"
)


for path in [
    STEP26A_OBJECT,
    MS3_MASTER_INPUT,
    AXIS_GENE_INPUT,
    RESIDUAL_INPUT,
    PROGRAM_INPUT,
    CORE50_INPUT,
    CORE100_INPUT
]:
    assert path.exists(), f"Missing file:\n{path}"


# ============================================================
# Load
# ============================================================

adata = sc.read_h5ad(
    STEP26A_OBJECT
)

ms3_gene_master = pd.read_csv(
    MS3_MASTER_INPUT,
    index_col=0
)

axis_gene_master = pd.read_csv(
    AXIS_GENE_INPUT,
    index_col=0
)

residual_ms3 = pd.read_csv(
    RESIDUAL_INPUT,
    index_col=0
)

master_program = pd.read_csv(
    PROGRAM_INPUT
)

core50_table = pd.read_csv(
    CORE50_INPUT,
    index_col=0
)

core100_table = pd.read_csv(
    CORE100_INPUT,
    index_col=0
)


def extract_gene_column(df):

    for col in ["Gene", "gene", "names"]:
        if col in df.columns:
            return df[col].astype(str).tolist()

    return df.index.astype(str).tolist()


MS3_CORE50 = extract_gene_column(
    core50_table
)

MS3_CORE100 = extract_gene_column(
    core100_table
)


STATE_KEY = "final_malignant_state_id"

MS3_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

COMMON_AXIS_PATIENTS = [
    "HCC09",
    "HCC10"
]


print("=" * 120)
print("STEP26B PATHWAY AND VULNERABILITY ANALYSIS")
print("=" * 120)

print("Single-cell object:", adata.shape)
print("MS3 gene master:", ms3_gene_master.shape)
print("MS3-MS6 axis:", axis_gene_master.shape)
print("Frozen Core50:", len(MS3_CORE50))
print("Frozen Core100:", len(MS3_CORE100))

print(
    "\nStep26B will NOT redefine MS3, MS6, Core50, "
    "or any previously frozen malignant-state labels."
)

In [ ]:
# ============================================================
# STEP26B - CELL 24
# Publication plotting
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False


mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


def save_publication_figure(
    fig,
    filename
):

    png = (
        FIGURES_DIR
        / f"{filename}.png"
    )

    pdf = (
        FIGURES_DIR
        / f"{filename}.pdf"
    )

    # Save BEFORE show
    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight",
        facecolor="white"
    )

    print("[OK]", png.name)
    print("[OK]", pdf.name)

In [ ]:
# ============================================================
# STEP26B - CELL 25
# Check / install GSEApy
# ============================================================

import importlib.util
import subprocess


if importlib.util.find_spec("gseapy") is None:

    print("GSEApy not found.")
    print("Installing GSEApy with live progress...\n")

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "gseapy",
        "--no-cache-dir",
        "--timeout",
        "120"
    ]

    print(" ".join(cmd))
    print("=" * 100)

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(
            "GSEApy installation failed."
        )


import gseapy as gp


print("\nGSEApy version:")
print(
    getattr(
        gp,
        "__version__",
        "version not reported"
    )
)

In [ ]:
# ============================================================
# STEP26B - CELL 26
# Retrieve and FREEZE external gene-set libraries
# ============================================================

LIBRARY_REQUESTS = {

    "Hallmark":
        [
            "MSigDB_Hallmark_2020"
        ],

    "KEGG":
        [
            "KEGG_2021_Human",
            "KEGG_2019_Human"
        ],

    "Reactome":
        [
            "Reactome_2022",
            "Reactome_2016"
        ],

    "GO_BP":
        [
            "GO_Biological_Process_2023",
            "GO_Biological_Process_2021",
            "GO_Biological_Process_2018"
        ]
}


def save_gene_sets_as_gmt(
    gene_sets,
    path
):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        for term, genes in gene_sets.items():

            line = (
                term
                + "\tNA\t"
                + "\t".join(
                    sorted(
                        set(
                            map(
                                str,
                                genes
                            )
                        )
                    )
                )
                + "\n"
            )

            f.write(line)


def read_gmt(
    path
):

    output = {}

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            parts = (
                line.rstrip("\n")
                .split("\t")
            )

            if len(parts) < 3:
                continue

            output[
                parts[0]
            ] = parts[2:]

    return output


# ============================================================
# Get available Enrichr libraries only if needed
# ============================================================

need_online = False

for label in LIBRARY_REQUESTS:

    frozen_path = (
        GENESET_DIR
        / f"{label}.gmt"
    )

    if not frozen_path.exists():
        need_online = True


available_libraries = None


if need_online:

    print(
        "At least one local GMT snapshot is missing."
    )

    print(
        "Querying Enrichr library catalogue..."
    )

    try:

        available_libraries = (
            gp.get_library_name(
                organism="Human"
            )
        )

    except Exception as e:

        raise RuntimeError(
            "Unable to access Enrichr. "
            "Check network/VPN and rerun Cell 26.\n"
            f"Original error: {e}"
        )


frozen_gene_sets = {}
library_manifest_rows = []


for label, preferred_names in (
    LIBRARY_REQUESTS.items()
):

    frozen_path = (
        GENESET_DIR
        / f"{label}.gmt"
    )


    # --------------------------------------------------------
    # Use frozen local copy if already available
    # --------------------------------------------------------

    if frozen_path.exists():

        gene_sets = read_gmt(
            frozen_path
        )

        frozen_gene_sets[
            label
        ] = gene_sets

        library_manifest_rows.append({

            "Library_label":
                label,

            "Resolved_library":
                "LOCAL_FROZEN_GMT",

            "N_gene_sets":
                len(gene_sets),

            "GMT_path":
                str(frozen_path)
        })

        print(
            f"[LOCAL] {label}: "
            f"{len(gene_sets)} gene sets"
        )

        continue


    # --------------------------------------------------------
    # Resolve online library
    # --------------------------------------------------------

    resolved = None

    for candidate in preferred_names:

        if candidate in available_libraries:

            resolved = candidate
            break


    if resolved is None:

        keywords = {

            "Hallmark":
                ["hallmark"],

            "KEGG":
                ["kegg", "human"],

            "Reactome":
                ["reactome"],

            "GO_BP":
                ["go_biological_process"]
        }


        candidates = [

            lib
            for lib in available_libraries

            if all(
                key.lower()
                in
                lib.lower()

                for key
                in keywords[label]
            )
        ]


        if len(candidates) == 0:

            raise RuntimeError(
                f"No suitable Enrichr library found for {label}."
            )


        resolved = sorted(
            candidates
        )[-1]


    print(
        f"[ONLINE] {label} -> {resolved}"
    )


    gene_sets = gp.get_library(

        name=
            resolved,

        organism=
            "Human"
    )


    frozen_gene_sets[
        label
    ] = gene_sets


    save_gene_sets_as_gmt(

        gene_sets,
        frozen_path
    )


    library_manifest_rows.append({

        "Library_label":
            label,

        "Resolved_library":
            resolved,

        "N_gene_sets":
            len(gene_sets),

        "GMT_path":
            str(frozen_path)
    })


gene_set_manifest = pd.DataFrame(
    library_manifest_rows
)


GENESET_MANIFEST_PATH = (

    REFERENCE_DIR
    / "Step26B_GeneSet_Library_Manifest.csv"
)


gene_set_manifest.to_csv(

    GENESET_MANIFEST_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("\nFrozen gene-set libraries:")

display(
    gene_set_manifest
)

In [ ]:
# ============================================================
# STEP26B - CELL 27
# Build ranked gene lists for preranked GSEA
# ============================================================

# ============================================================
# MS3 vs same-patient other malignant states
# ============================================================

ms3_rank = (
    ms3_gene_master
    .copy()
)


ms3_rank = ms3_rank[
    (
        ~ms3_rank[
            "Is_MT"
        ].astype(bool)
    )

    &

    (
        ~ms3_rank[
            "Is_Ribosomal"
        ].astype(bool)
    )
].copy()


ms3_rank[
    "Rank_metric"
] = (

    ms3_rank[
        "Median_delta"
    ]
    .astype(float)
)


ms3_rank_table = (

    ms3_rank[
        [
            "Rank_metric",
            "Mean_delta",
            "Positive_fraction",
            "Negative_fraction"
        ]
    ]

    .sort_values(
        "Rank_metric",
        ascending=False
    )

    .reset_index()
)


ms3_rank_table.columns = [

    "Gene",
    "Rank_metric",
    "Mean_delta",
    "Positive_fraction",
    "Negative_fraction"
]


MS3_RANK_PATH = (

    RESULTS_DIR
    / "Step26B_MS3_Consensus_RankedGenes.csv"
)


ms3_rank_table.to_csv(

    MS3_RANK_PATH,

    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# MS3 versus MS6 paired axis
# ============================================================

axis_rank = (
    axis_gene_master
    .copy()
)


axis_rank = axis_rank[
    ~axis_rank.index
    .str.upper()
    .str.startswith(
        "MT-"
    )
].copy()


axis_rank = axis_rank[
    ~axis_rank.index
    .str.upper()
    .str.match(
        r"^RP[SL]"
    )
].copy()


axis_rank[
    "Rank_metric"
] = (
    axis_rank[
        "Mean_delta"
    ]
    .astype(float)
)


axis_rank_table = (

    axis_rank[
        [
            "Rank_metric",
            "Min_delta",
            "Max_delta",
            "Direction"
        ]
    ]

    .sort_values(
        "Rank_metric",
        ascending=False
    )

    .reset_index()
)


axis_rank_table.columns = [

    "Gene",
    "Rank_metric",
    "Min_delta",
    "Max_delta",
    "Direction"
]


AXIS_RANK_PATH = (

    RESULTS_DIR
    / "Step26B_MS3_vs_MS6_RankedGenes.csv"
)


axis_rank_table.to_csv(

    AXIS_RANK_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "MS3 consensus ranked genes:",
    len(ms3_rank_table)
)

print(
    "MS3-MS6 ranked genes:",
    len(axis_rank_table)
)

In [ ]:
# ============================================================
# STEP26B - CELL 28
# Preranked GSEA
# ============================================================

PERMUTATION_NUM = 1000


def clean_gsea_result(
    res,
    analysis,
    library
):

    df = res.copy()


    if "Term" not in df.columns:

        df = (
            df.reset_index()
        )


        if "Term" not in df.columns:

            first_col = df.columns[0]

            df = df.rename(
                columns={
                    first_col:
                        "Term"
                }
            )


    # Standardize numeric columns
    rename_map = {

        "NOM p-val":
            "P_value",

        "FDR q-val":
            "FDR",

        "Lead_genes":
            "Leading_edge",

        "ledge_genes":
            "Leading_edge"
    }


    df = df.rename(
        columns=
            rename_map
    )


    for col in [
        "ES",
        "NES",
        "P_value",
        "FDR"
    ]:

        if col in df.columns:

            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )


    df.insert(
        0,
        "Analysis",
        analysis
    )

    df.insert(
        1,
        "Library",
        library
    )


    return df


def run_prerank(
    rank_table,
    analysis_name
):

    all_results = []


    rnk = (
        rank_table[
            [
                "Gene",
                "Rank_metric"
            ]
        ]
        .dropna()
        .drop_duplicates(
            "Gene"
        )
    )


    for library_label, gene_sets in (
        frozen_gene_sets.items()
    ):

        print(
            f"Running {analysis_name} | {library_label} ..."
        )


        try:

            pre = gp.prerank(

                rnk=
                    rnk,

                gene_sets=
                    gene_sets,

                min_size=
                    10,

                max_size=
                    500,

                permutation_num=
                    PERMUTATION_NUM,

                seed=
                    SEED,

                threads=
                    4,

                outdir=
                    None,

                verbose=
                    False
            )


            result = clean_gsea_result(

                pre.res2d,

                analysis=
                    analysis_name,

                library=
                    library_label
            )


            all_results.append(
                result
            )


        except Exception as e:

            print(
                f"[FAILED] {analysis_name} | "
                f"{library_label}: {e}"
            )


    if len(all_results) == 0:

        return pd.DataFrame()


    return pd.concat(
        all_results,
        ignore_index=True
    )


# ============================================================
# Run
# ============================================================

gsea_ms3 = run_prerank(

    ms3_rank_table,

    "MS3_vs_OtherMalignantStates"
)


gsea_axis = run_prerank(

    axis_rank_table,

    "MS3_vs_MS6"
)


GSEA_MS3_PATH = (

    RESULTS_DIR
    / "Step26B_GSEA_MS3_vs_OtherStates.csv"
)


GSEA_AXIS_PATH = (

    RESULTS_DIR
    / "Step26B_GSEA_MS3_vs_MS6.csv"
)


gsea_ms3.to_csv(

    GSEA_MS3_PATH,

    index=False,
    encoding="utf-8-sig"
)


gsea_axis.to_csv(

    GSEA_AXIS_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "\nMS3 GSEA rows:",
    len(gsea_ms3)
)

print(
    "MS3-MS6 GSEA rows:",
    len(gsea_axis)
)

In [ ]:
# ============================================================
# STEP26B - CELL 29
# Define consistent ORA query sets
# ============================================================

MS3_UP_TOP500 = (

    ms3_gene_master[
        (
            ms3_gene_master[
                "Direction"
            ]
            ==
            "Up_in_all_MS3_patients"
        )

        &

        (
            ~ms3_gene_master[
                "Is_MT"
            ].astype(bool)
        )

        &

        (
            ~ms3_gene_master[
                "Is_Ribosomal"
            ].astype(bool)
        )
    ]

    .sort_values(
        [
            "Min_delta",
            "Mean_delta"
        ],
        ascending=[
            False,
            False
        ]
    )

    .head(500)

    .index
    .astype(str)
    .tolist()
)


MS3_DOWN_TOP500 = (

    ms3_gene_master[
        (
            ms3_gene_master[
                "Direction"
            ]
            ==
            "Down_in_all_MS3_patients"
        )

        &

        (
            ~ms3_gene_master[
                "Is_MT"
            ].astype(bool)
        )

        &

        (
            ~ms3_gene_master[
                "Is_Ribosomal"
            ].astype(bool)
        )
    ]

    .sort_values(
        [
            "Max_delta",
            "Mean_delta"
        ],
        ascending=[
            True,
            True
        ]
    )

    .head(500)

    .index
    .astype(str)
    .tolist()
)


AXIS_MS3_TOP500 = (

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS3_higher_in_both_patients"
    ]

    .sort_values(
        "Min_delta",
        ascending=False
    )

    .head(500)

    .index
    .astype(str)
    .tolist()
)


AXIS_MS6_TOP500 = (

    axis_gene_master[
        axis_gene_master[
            "Direction"
        ]
        ==
        "MS6_higher_in_both_patients"
    ]

    .sort_values(
        "Max_delta",
        ascending=True
    )

    .head(500)

    .index
    .astype(str)
    .tolist()
)


print(
    "MS3 consistent up:",
    len(MS3_UP_TOP500)
)

print(
    "MS3 consistent down:",
    len(MS3_DOWN_TOP500)
)

print(
    "MS3 > MS6:",
    len(AXIS_MS3_TOP500)
)

print(
    "MS6 > MS3:",
    len(AXIS_MS6_TOP500)
)

In [ ]:
# ============================================================
# STEP26B - CELL 30
# Local hypergeometric ORA
# ============================================================

def bh_adjust(
    pvalues
):

    pvalues = np.asarray(
        pvalues,
        dtype=float
    )

    n = len(
        pvalues
    )

    if n == 0:
        return pvalues


    order = np.argsort(
        pvalues
    )

    ranked = (
        pvalues[
            order
        ]
    )


    adjusted = (

        ranked
        *
        n
        /
        np.arange(
            1,
            n + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]


    adjusted = np.minimum(
        adjusted,
        1.0
    )


    output = np.empty(
        n
    )

    output[
        order
    ] = adjusted


    return output


def run_ora(
    query_genes,
    query_name,
    background
):

    query = (
        set(
            map(
                str,
                query_genes
            )
        )
        &
        background
    )


    M = len(
        background
    )

    N = len(
        query
    )


    output_rows = []


    for library_label, gene_sets in (
        frozen_gene_sets.items()
    ):

        local_rows = []


        for pathway, genes in (
            gene_sets.items()
        ):

            genes = (
                set(
                    map(
                        str,
                        genes
                    )
                )
                &
                background
            )


            K = len(
                genes
            )


            overlap = (
                query
                &
                genes
            )


            k = len(
                overlap
            )


            if k < 3:
                continue


            p_value = hypergeom.sf(

                k - 1,
                M,
                K,
                N
            )


            local_rows.append({

                "Query":
                    query_name,

                "Library":
                    library_label,

                "Pathway":
                    pathway,

                "Query_size":
                    N,

                "Pathway_size":
                    K,

                "Overlap":
                    k,

                "Overlap_fraction":
                    k / N,

                "P_value":
                    p_value,

                "Genes":
                    "|".join(
                        sorted(
                            overlap
                        )
                    )
            })


        if len(local_rows) == 0:
            continue


        local_df = pd.DataFrame(
            local_rows
        )


        local_df[
            "FDR"
        ] = bh_adjust(
            local_df[
                "P_value"
            ].values
        )


        output_rows.append(
            local_df
        )


    if len(output_rows) == 0:

        return pd.DataFrame()


    return (

        pd.concat(
            output_rows,
            ignore_index=True
        )

        .sort_values(
            [
                "FDR",
                "P_value"
            ]
        )
    )


background = set(
    adata.var_names.astype(str)
)


ora_queries = {

    "MS3_Consistent_Up_Top500":
        MS3_UP_TOP500,

    "MS3_Consistent_Down_Top500":
        MS3_DOWN_TOP500,

    "MS3_vs_MS6_Up_Top500":
        AXIS_MS3_TOP500,

    "MS6_vs_MS3_Up_Top500":
        AXIS_MS6_TOP500
}


ora_results = []


for name, genes in (
    ora_queries.items()
):

    print(
        "ORA:",
        name
    )


    temp = run_ora(

        query_genes=
            genes,

        query_name=
            name,

        background=
            background
    )


    if len(temp) > 0:

        ora_results.append(
            temp
        )


if len(ora_results) > 0:

    ora_all = pd.concat(
        ora_results,
        ignore_index=True
    )

else:

    ora_all = pd.DataFrame()


ORA_ALL_PATH = (

    RESULTS_DIR
    / "Step26B_ORA_AllQueries.csv"
)


ora_all.to_csv(

    ORA_ALL_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "ORA rows:",
    len(ora_all)
)

In [ ]:
# ============================================================
# STEP26B - CELL 31
# Construct operational cell-cycle blacklist
# ============================================================

MANUAL_MITOTIC_GENES = {

    "CDK1", "CCNB1", "CCNB2", "CDC25C",
    "PLK1", "AURKA", "AURKB", "BORA",

    "CENPA", "CENPE", "CENPF", "KIF2C",
    "KIF11", "KIF20A", "BUB1", "BUB1B",
    "MAD2L1", "NDC80", "NUF2", "SPC25",

    "PRC1", "KIF23", "KIF20B", "ECT2",
    "RACGAP1", "ANLN", "CEP55",

    "TPX2", "NEK2", "MKI67",
    "CDC20", "TTK", "DLGAP5",
    "NUSAP1", "PTTG1", "UBE2C"
}


hallmark_sets = (
    frozen_gene_sets[
        "Hallmark"
    ]
)


hallmark_cycle_terms = [

    term
    for term in hallmark_sets

    if (
        "G2M" in term.upper()
        or
        "E2F" in term.upper()
    )
]


hallmark_cycle_genes = set()


for term in hallmark_cycle_terms:

    hallmark_cycle_genes.update(
        hallmark_sets[
            term
        ]
    )


CELL_CYCLE_BLACKLIST = (

    set(
        MS3_CORE100
    )

    |

    MANUAL_MITOTIC_GENES

    |

    hallmark_cycle_genes
)


BLACKLIST_PATH = (

    REFERENCE_DIR
    / "Step26B_CellCycle_Blacklist.txt"
)


with open(
    BLACKLIST_PATH,
    "w",
    encoding="utf-8"
) as f:

    for gene in sorted(
        CELL_CYCLE_BLACKLIST
    ):

        f.write(
            gene
            +
            "\n"
        )


print(
    "Hallmark cycle terms:",
    hallmark_cycle_terms
)

print(
    "Operational cell-cycle blacklist genes:",
    len(
        CELL_CYCLE_BLACKLIST
    )
)

In [ ]:
# ============================================================
# STEP26B - CELL 32
# Conserved MS3 residual biology after cycle-gene exclusion
# ============================================================

ms3_residual_noncycle = (

    ms3_gene_master[
        (
            ms3_gene_master[
                "Direction"
            ]
            ==
            "Up_in_all_MS3_patients"
        )

        &

        (
            ~ms3_gene_master.index
            .isin(
                CELL_CYCLE_BLACKLIST
            )
        )

        &

        (
            ~ms3_gene_master[
                "Is_MT"
            ].astype(bool)
        )

        &

        (
            ~ms3_gene_master[
                "Is_Ribosomal"
            ].astype(bool)
        )
    ]

    .sort_values(
        [
            "Min_delta",
            "Mean_delta"
        ],
        ascending=[
            False,
            False
        ]
    )

    .copy()
)


RESIDUAL_NONCYCLE_PATH = (

    RESULTS_DIR
    / "Step26B_MS3_Residual_NonCellCycle_Genes.csv"
)


ms3_residual_noncycle.to_csv(

    RESIDUAL_NONCYCLE_PATH,

    encoding="utf-8-sig"
)


print("=" * 110)
print("RESIDUAL NON-CELL-CYCLE MS3 BIOLOGY")
print("=" * 110)

print(
    "Consistent MS3-up genes after operational "
    "cell-cycle exclusion:",
    len(
        ms3_residual_noncycle
    )
)


display(
    ms3_residual_noncycle
    .head(60)
    .round(4)
)

In [ ]:
# ============================================================
# STEP26B - CELL 33
# Preranked GSEA after operational cell-cycle exclusion
# ============================================================

residual_rank = (

    ms3_rank_table[
        ~ms3_rank_table[
            "Gene"
        ]
        .isin(
            CELL_CYCLE_BLACKLIST
        )
    ]

    .copy()
)


RESIDUAL_RANK_PATH = (

    RESULTS_DIR
    / "Step26B_MS3_NonCellCycle_RankedGenes.csv"
)


residual_rank.to_csv(

    RESIDUAL_RANK_PATH,

    index=False,
    encoding="utf-8-sig"
)


gsea_residual = run_prerank(

    residual_rank,

    "MS3_NonCellCycleResidual"
)


GSEA_RESIDUAL_PATH = (

    RESULTS_DIR
    / "Step26B_GSEA_MS3_NonCellCycleResidual.csv"
)


gsea_residual.to_csv(

    GSEA_RESIDUAL_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Residual ranked genes:",
    len(
        residual_rank
    )
)

print(
    "Residual GSEA rows:",
    len(
        gsea_residual
    )
)

In [ ]:
# ============================================================
# STEP26B - CELL 34
# Residual ORA sensitivity analysis
# ============================================================

residual_ora_results = []


for top_n in [
    300,
    500
]:

    query = (

        ms3_residual_noncycle
        .head(
            top_n
        )
        .index
        .astype(str)
        .tolist()
    )


    temp = run_ora(

        query_genes=
            query,

        query_name=
            f"MS3_NonCellCycle_Top{top_n}",

        background=
            background
    )


    if len(temp) > 0:

        residual_ora_results.append(
            temp
        )


if len(residual_ora_results) > 0:

    ora_residual = pd.concat(

        residual_ora_results,

        ignore_index=True
    )

else:

    ora_residual = pd.DataFrame()


ORA_RESIDUAL_PATH = (

    RESULTS_DIR
    / "Step26B_ORA_MS3_NonCellCycleResidual.csv"
)


ora_residual.to_csv(

    ORA_RESIDUAL_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Residual ORA rows:",
    len(
        ora_residual
    )
)

In [ ]:
# ============================================================
# STEP26B - CELL 35
# Significant GSEA summary
# ============================================================

def significant_gsea(
    df,
    fdr_cutoff=0.05
):

    if len(df) == 0:
        return df.copy()


    temp = df.copy()


    if "FDR" not in temp.columns:

        raise KeyError(
            "FDR column missing."
        )


    return (

        temp[
            temp[
                "FDR"
            ]
            <=
            fdr_cutoff
        ]

        .sort_values(
            [
                "Library",
                "NES"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


sig_gsea_ms3 = significant_gsea(
    gsea_ms3
)

sig_gsea_axis = significant_gsea(
    gsea_axis
)

sig_gsea_residual = significant_gsea(
    gsea_residual
)


SIG_GSEA_PATH = (

    RESULTS_DIR
    / "Step26B_GSEA_Significant_Master.csv"
)


sig_gsea_master = pd.concat(

    [
        sig_gsea_ms3,
        sig_gsea_axis,
        sig_gsea_residual
    ],

    ignore_index=True
)


sig_gsea_master.to_csv(

    SIG_GSEA_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Significant MS3 GSEA terms:",
    len(
        sig_gsea_ms3
    )
)

print(
    "Significant MS3-MS6 terms:",
    len(
        sig_gsea_axis
    )
)

print(
    "Significant residual terms:",
    len(
        sig_gsea_residual
    )
)


for analysis_name, df in [

    (
        "MS3 vs other malignant states",
        sig_gsea_ms3
    ),

    (
        "MS3 vs MS6",
        sig_gsea_axis
    ),

    (
        "Non-cell-cycle residual",
        sig_gsea_residual
    )
]:

    print(
        "\n",
        analysis_name
    )

    if len(df) == 0:

        print(
            "No FDR <= 0.05 terms."
        )

        continue


    display(

        df[
            [
                col
                for col in [
                    "Library",
                    "Term",
                    "NES",
                    "P_value",
                    "FDR",
                    "Leading_edge"
                ]
                if col in df.columns
            ]
        ]

        .head(40)

        .round(4)
    )

In [ ]:
# ============================================================
# STEP26B - CELL 36
# Main MS3 enrichment figure
# ============================================================

def select_gsea_terms_for_plot(
    df,
    top_positive=4,
    top_negative=4
):

    selected = []


    for library in [
        "Hallmark",
        "KEGG",
        "Reactome",
        "GO_BP"
    ]:

        temp = df[
            df[
                "Library"
            ]
            ==
            library
        ].copy()


        if len(temp) == 0:
            continue


        pos = (

            temp[
                temp[
                    "NES"
                ]
                >
                0
            ]

            .sort_values(
                [
                    "FDR",
                    "NES"
                ],
                ascending=[
                    True,
                    False
                ]
            )

            .head(
                top_positive
            )
        )


        neg = (

            temp[
                temp[
                    "NES"
                ]
                <
                0
            ]

            .sort_values(
                [
                    "FDR",
                    "NES"
                ],
                ascending=[
                    True,
                    True
                ]
            )

            .head(
                top_negative
            )
        )


        selected.append(
            pos
        )

        selected.append(
            neg
        )


    if len(selected) == 0:

        return pd.DataFrame()


    output = pd.concat(
        selected,
        ignore_index=True
    )


    return output.drop_duplicates(
        [
            "Library",
            "Term"
        ]
    )


main_plot_df = select_gsea_terms_for_plot(
    sig_gsea_ms3
)


if len(main_plot_df) > 0:

    main_plot_df[
        "MinusLog10FDR"
    ] = (

        -np.log10(

            main_plot_df[
                "FDR"
            ]
            .clip(
                lower=1e-300
            )
        )
    )


    main_plot_df[
        "Display_term"
    ] = (

        main_plot_df[
            "Term"
        ]
        .astype(str)
        .str.replace(
            "_",
            " ",
            regex=False
        )
    )


    main_plot_df = (
        main_plot_df
        .sort_values(
            "NES"
        )
        .reset_index(
            drop=True
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            9.5,
            max(
                6.5,
                len(
                    main_plot_df
                )
                * 0.27
            )
        )
    )


    y = np.arange(
        len(
            main_plot_df
        )
    )


    sizes = (

        20
        +
        main_plot_df[
            "MinusLog10FDR"
        ].values
        *
        18
    )


    scatter = ax.scatter(

        main_plot_df[
            "NES"
        ],

        y,

        s=
            sizes,

        c=
            main_plot_df[
                "NES"
            ],

        cmap=
            "RdBu_r",

        vmin=
            -max(
                2.5,
                np.abs(
                    main_plot_df[
                        "NES"
                    ]
                ).max()
            ),

        vmax=
            max(
                2.5,
                np.abs(
                    main_plot_df[
                        "NES"
                    ]
                ).max()
            ),

        edgecolors=
            "black",

        linewidths=
            0.3
    )


    ax.axvline(
        0,
        linewidth=0.8
    )


    ax.set_yticks(
        y
    )


    ax.set_yticklabels(
        main_plot_df[
            "Display_term"
        ]
    )


    ax.set_xlabel(
        "Normalized Enrichment Score"
    )

    ax.set_ylabel(
        "Gene Set"
    )

    ax.set_title(
        "Consensus Pathway Enrichment of the MS3 Malignant State"
    )


    cbar = fig.colorbar(
        scatter,
        ax=ax
    )

    cbar.set_label(
        "Normalized Enrichment Score"
    )


    fig.tight_layout()


    save_publication_figure(

        fig,

        "Step26B_01_MS3_Consensus_GSEA"
    )


    plt.show()
    plt.close(fig)


else:

    print(
        "No significant GSEA terms available for plotting."
    )

In [ ]:
# ============================================================
# STEP26B - CELL 37
# Non-cell-cycle residual enrichment figure
# ============================================================

residual_plot_df = (
    select_gsea_terms_for_plot(
        sig_gsea_residual,
        top_positive=5,
        top_negative=5
    )
)


if len(residual_plot_df) > 0:

    residual_plot_df[
        "MinusLog10FDR"
    ] = (

        -np.log10(

            residual_plot_df[
                "FDR"
            ]
            .clip(
                lower=1e-300
            )
        )
    )


    residual_plot_df[
        "Display_term"
    ] = (

        residual_plot_df[
            "Term"
        ]
        .astype(str)
        .str.replace(
            "_",
            " ",
            regex=False
        )
    )


    residual_plot_df = (

        residual_plot_df
        .sort_values(
            "NES"
        )
        .reset_index(
            drop=True
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            9.5,
            max(
                6.5,
                len(
                    residual_plot_df
                )
                * 0.27
            )
        )
    )


    y = np.arange(
        len(
            residual_plot_df
        )
    )


    sizes = (

        20
        +
        residual_plot_df[
            "MinusLog10FDR"
        ].values
        *
        18
    )


    lim = max(

        2.5,

        np.abs(
            residual_plot_df[
                "NES"
            ]
        ).max()
    )


    scatter = ax.scatter(

        residual_plot_df[
            "NES"
        ],

        y,

        s=
            sizes,

        c=
            residual_plot_df[
                "NES"
            ],

        cmap=
            "RdBu_r",

        vmin=
            -lim,

        vmax=
            lim,

        edgecolors=
            "black",

        linewidths=
            0.3
    )


    ax.axvline(
        0,
        linewidth=0.8
    )


    ax.set_yticks(
        y
    )


    ax.set_yticklabels(
        residual_plot_df[
            "Display_term"
        ]
    )


    ax.set_xlabel(
        "Normalized Enrichment Score"
    )

    ax.set_ylabel(
        "Gene Set"
    )

    ax.set_title(
        "Residual MS3 Biology After Cell-cycle Gene Exclusion"
    )


    cbar = fig.colorbar(
        scatter,
        ax=ax
    )


    cbar.set_label(
        "Normalized Enrichment Score"
    )


    fig.tight_layout()


    save_publication_figure(

        fig,

        "Step26B_02_MS3_NonCellCycle_Residual_GSEA"
    )


    plt.show()
    plt.close(fig)


else:

    print(
        "No significant residual GSEA terms available for plotting."
    )

In [ ]:
# ============================================================
# STEP26B - CELL 38
# Evidence table for candidate mechanistic vulnerabilities
# ============================================================

VULNERABILITY_PANEL = {

    # Mitotic control
    "PLK1":
        "Mitotic kinase",

    "AURKA":
        "Mitotic kinase",

    "AURKB":
        "Chromosome segregation / cytokinesis",

    "TTK":
        "Spindle checkpoint",

    "NEK2":
        "Mitotic / centrosomal kinase",

    "CDK1":
        "Mitotic entry",

    # Spindle checkpoint
    "BUB1":
        "Spindle checkpoint",

    "BUB1B":
        "Spindle checkpoint",

    "KIF11":
        "Mitotic spindle",

    "KIF2C":
        "Chromosome segregation",

    "CENPE":
        "Chromosome segregation",

    # Replication stress
    "ATR":
        "Replication-stress checkpoint",

    "CHEK1":
        "Replication-stress checkpoint",

    "WEE1":
        "Replication-stress checkpoint",

    "CLSPN":
        "Replication-stress checkpoint",

    "RRM2":
        "Nucleotide supply / replication stress",

    # DNA repair
    "RAD51":
        "Homologous recombination",

    "FANCD2":
        "Fanconi DNA repair",

    "UBE2T":
        "Fanconi DNA repair",

    "BRCA1":
        "Homologous recombination",

    # Survival
    "BIRC5":
        "Mitotic survival / apoptosis resistance",

    "BCL2L1":
        "Apoptosis resistance",

    "MCL1":
        "Apoptosis resistance",

    # DNA topology
    "TOP2A":
        "DNA topology / mitosis"
}


def detection_fraction_by_patient(
    gene,
    target_state="MS3"
):

    results = {}


    if gene not in adata.var_names:

        for patient in MS3_PATIENTS:
            results[
                patient
            ] = np.nan

        return results


    for patient in MS3_PATIENTS:

        mask = (

            (
                adata.obs[
                    "patient"
                ]
                .astype(str)
                ==
                patient
            )

            &

            (
                adata.obs[
                    STATE_KEY
                ]
                .astype(str)
                ==
                target_state
            )
        )


        X_gene = (
            adata[
                mask,
                gene
            ]
            .X
        )


        if sparse.issparse(
            X_gene
        ):

            detected = (
                X_gene.toarray()
                .ravel()
                >
                0
            )

        else:

            detected = (
                np.asarray(
                    X_gene
                )
                .ravel()
                >
                0
            )


        results[
            patient
        ] = float(
            np.mean(
                detected
            )
        )


    return results


vulnerability_rows = []


for gene, category in (
    VULNERABILITY_PANEL.items()
):

    if gene not in ms3_gene_master.index:
        continue


    detect = (
        detection_fraction_by_patient(
            gene
        )
    )


    axis_present = (
        gene
        in
        axis_gene_master.index
    )


    axis_mean = (

        float(
            axis_gene_master.loc[
                gene,
                "Mean_delta"
            ]
        )

        if axis_present

        else np.nan
    )


    axis_direction = (

        str(
            axis_gene_master.loc[
                gene,
                "Direction"
            ]
        )

        if axis_present

        else "Not_available"
    )


    row = {

        "Gene":
            gene,

        "Mechanistic_category":
            category,

        "MS3_Mean_delta":
            float(
                ms3_gene_master.loc[
                    gene,
                    "Mean_delta"
                ]
            ),

        "MS3_Min_delta":
            float(
                ms3_gene_master.loc[
                    gene,
                    "Min_delta"
                ]
            ),

        "MS3_Positive_fraction":
            float(
                ms3_gene_master.loc[
                    gene,
                    "Positive_fraction"
                ]
            ),

        "MS3_vs_MS6_Mean_delta":
            axis_mean,

        "MS3_vs_MS6_Direction":
            axis_direction,

        "Detection_HCC08":
            detect[
                "HCC08"
            ],

        "Detection_HCC09":
            detect[
                "HCC09"
            ],

        "Detection_HCC10":
            detect[
                "HCC10"
            ],

        "Min_detection":
            float(
                np.nanmin(
                    list(
                        detect.values()
                    )
                )
            ),

        "Is_Core50":
            gene
            in
            MS3_CORE50,

        "Is_Core100":
            gene
            in
            MS3_CORE100
    }


    # --------------------------------------------------------
    # Transparent evidence count
    # NOT a therapeutic efficacy score
    # --------------------------------------------------------

    evidence_count = 0


    if (
        row[
            "MS3_Positive_fraction"
        ]
        ==
        1.0
    ):
        evidence_count += 1


    if (
        row[
            "MS3_Min_delta"
        ]
        >
        0
    ):
        evidence_count += 1


    if (
        row[
            "MS3_vs_MS6_Direction"
        ]
        ==
        "MS3_higher_in_both_patients"
    ):
        evidence_count += 1


    if (
        row[
            "Min_detection"
        ]
        >=
        0.10
    ):
        evidence_count += 1


    if (
        row[
            "Is_Core50"
        ]
        or
        row[
            "Is_Core100"
        ]
    ):
        evidence_count += 1


    row[
        "Evidence_count"
    ] = evidence_count


    vulnerability_rows.append(
        row
    )


vulnerability_table = (

    pd.DataFrame(
        vulnerability_rows
    )

    .sort_values(
        [
            "Evidence_count",
            "MS3_Min_delta",
            "MS3_vs_MS6_Mean_delta"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)


VULNERABILITY_PATH = (

    RESULTS_DIR
    / "Step26B_Candidate_Vulnerability_Evidence.csv"
)


vulnerability_table.to_csv(

    VULNERABILITY_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    vulnerability_table
    .round(4)
)

In [ ]:
# ============================================================
# STEP26B - CELL 39
# Candidate vulnerability evidence heatmap
# ============================================================

top_candidates = (

    vulnerability_table[
        vulnerability_table[
            "MS3_Positive_fraction"
        ]
        ==
        1.0
    ]

    .head(18)

    .copy()
)


plot_rows = []


for _, row in (
    top_candidates.iterrows()
):

    gene = row[
        "Gene"
    ]


    plot_rows.append({

        "Gene":
            gene,

        "HCC08 MS3 Delta":
            ms3_gene_master.loc[
                gene,
                "Delta_HCC08"
            ],

        "HCC09 MS3 Delta":
            ms3_gene_master.loc[
                gene,
                "Delta_HCC09"
            ],

        "HCC10 MS3 Delta":
            ms3_gene_master.loc[
                gene,
                "Delta_HCC10"
            ],

        "HCC09 MS3-MS6":
            (
                axis_gene_master.loc[
                    gene,
                    "Delta_HCC09"
                ]

                if gene
                in axis_gene_master.index

                else np.nan
            ),

        "HCC10 MS3-MS6":
            (
                axis_gene_master.loc[
                    gene,
                    "Delta_HCC10"
                ]

                if gene
                in axis_gene_master.index

                else np.nan
            )
    })


vulnerability_heat = (

    pd.DataFrame(
        plot_rows
    )

    .set_index(
        "Gene"
    )
)


VULNERABILITY_HEAT_PATH = (

    RESULTS_DIR
    / "Step26B_CandidateVulnerability_HeatmapData.csv"
)


vulnerability_heat.to_csv(
    VULNERABILITY_HEAT_PATH,
    encoding="utf-8-sig"
)


fig, ax = plt.subplots(
    figsize=(
        7.8,
        max(
            6.0,
            len(
                vulnerability_heat
            )
            * 0.38
        )
    )
)


values = (
    vulnerability_heat.values
)


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap=
        "RdBu_r",

    vmin=
        -lim,

    vmax=
        lim,

    aspect=
        "auto"
)


ax.set_xticks(
    np.arange(
        vulnerability_heat.shape[1]
    )
)


ax.set_xticklabels(

    vulnerability_heat.columns,

    rotation=35,

    ha="right"
)


ax.set_yticks(
    np.arange(
        vulnerability_heat.shape[0]
    )
)


ax.set_yticklabels(
    vulnerability_heat.index
)


ax.set_xlabel(
    "Cross-patient Evidence"
)

ax.set_ylabel(
    "Candidate Gene"
)

ax.set_title(
    "Cross-patient Evidence for Candidate MS3 Vulnerabilities"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Patient-aware Expression Delta"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step26B_03_Candidate_MS3_Vulnerabilities"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP26B - CELL 40
# Master interpretation table
# ============================================================

interpretation_rows = []


# ============================================================
# GSEA significant terms
# ============================================================

for analysis_name, df in [

    (
        "MS3_vs_OtherMalignantStates",
        sig_gsea_ms3
    ),

    (
        "MS3_vs_MS6",
        sig_gsea_axis
    ),

    (
        "MS3_NonCellCycleResidual",
        sig_gsea_residual
    )
]:

    if len(df) == 0:
        continue


    for _, row in (
        df.iterrows()
    ):

        interpretation_rows.append({

            "Evidence_type":
                "Preranked_GSEA",

            "Analysis":
                analysis_name,

            "Library":
                row.get(
                    "Library",
                    np.nan
                ),

            "Term":
                row.get(
                    "Term",
                    np.nan
                ),

            "Effect":
                row.get(
                    "NES",
                    np.nan
                ),

            "FDR":
                row.get(
                    "FDR",
                    np.nan
                ),

            "Direction":
                (
                    "Enriched_in_MS3"
                    if row.get(
                        "NES",
                        0
                    )
                    >
                    0

                    else
                    "Depleted_in_MS3"
                )
        })


step26b_master = pd.DataFrame(
    interpretation_rows
)


STEP26B_MASTER_PATH = (

    RESULTS_DIR
    / "Step26B_Master_Pathway_Evidence.csv"
)


step26b_master.to_csv(

    STEP26B_MASTER_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Master pathway evidence rows:",
    len(
        step26b_master
    )
)

In [ ]:
# ============================================================
# STEP26B - CELL 41
# Save Step26B checkpoint
# ============================================================

adata26b = (
    adata.copy()
)


adata26b.uns[
    "Step26B_pathway_and_vulnerability"
] = {

    "Status":
        "COMPLETE_PENDING_REVIEW",

    "Primary_state":
        "MS3",

    "Reference_state":
        "MS6",

    "Primary_signature":
        "MS3 Conserved Core50",

    "Gene_set_libraries":
        gene_set_manifest[
            "Library_label"
        ]
        .astype(str)
        .tolist(),

    "Permutation_number":
        int(
            PERMUTATION_NUM
        ),

    "Primary_enrichment_method":
        "Preranked GSEA",

    "Secondary_enrichment_method":
        "Hypergeometric ORA",

    "Residual_analysis":
        (
            "Operational cell-cycle exclusion using frozen Core100, "
            "Hallmark G2M/E2F sets and curated mitotic genes."
        ),

    "Important":
        (
            "Residual analysis is a gene-set exclusion sensitivity "
            "analysis. It is not cell-cycle regression and does not "
            "alter the frozen MS3 state."
        ),

    "Vulnerability_analysis":
        (
            "Evidence prioritization only; not drug-response "
            "prediction and not proof of therapeutic efficacy."
        ),

    "State_labels_modified":
        False,

    "Core50_modified":
        False
}


STEP26B_OBJECT = (

    DATA_DIR
    / "GSE149614_STEP26B_PATHWAY_VULNERABILITY_CHECKPOINT.h5ad"
)


adata26b.write_h5ad(

    STEP26B_OBJECT,

    compression="gzip"
)


print(
    "Saved:",
    STEP26B_OBJECT
)

In [ ]:
# ============================================================
# STEP26B - CELL 42
# Methodology / decision log
# ============================================================

step26b_summary = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step26B_Pathway_and_Vulnerability_Analysis",

    "Status":
        "COMPLETE_PENDING_REVIEW",

    "Primary_state":
        "MS3 Cycling-G2M",

    "Reference_state":
        "MS6 Hepatic-Differentiated",

    "Gene_set_libraries":
        gene_set_manifest.to_dict(
            orient="records"
        ),

    "Primary_rank_metric_MS3":
        (
            "Median within-patient pseudobulk expression delta "
            "across HCC08, HCC09 and HCC10."
        ),

    "Primary_rank_metric_axis":
        (
            "Mean paired MS3-minus-MS6 expression delta "
            "across HCC09 and HCC10."
        ),

    "GSEA_permutations":
        int(
            PERMUTATION_NUM
        ),

    "Residual_strategy":
        (
            "Exclude frozen MS3 Core100, Hallmark G2M/E2F genes, "
            "and curated mitotic genes from the ranked list."
        ),

    "Residual_strategy_interpretation":
        (
            "Used only to expose biology beyond the dominant "
            "cell-cycle program; not used to redefine MS3."
        ),

    "Candidate_vulnerability_strategy":
        (
            "Evidence table integrates three-patient MS3 consistency, "
            "paired MS3-vs-MS6 direction, expression detection and "
            "Core50/Core100 membership. It is hypothesis prioritization, "
            "not therapeutic validation."
        ),

    "Next_step":
        (
            "Review pathway and residual results. If coherent, "
            "freeze Step26 and proceed to Step27 trajectory analysis."
        )
}


SUMMARY_PATH = (

    LOGS_DIR
    / "Step26B_Final_Summary.json"
)


with open(

    SUMMARY_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step26b_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_PATH
)

In [ ]:
# ============================================================
# STEP26B - CELL 43
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR),
    ("Reference", REFERENCE_DIR),
    ("GeneSets", GENESET_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (

    RESULTS_DIR
    / "Step26B_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_PATH
)

In [ ]:
# ============================================================
# STEP26B - CELL 44
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 130)
print("STEP26B FINAL PATHWAY AND VULNERABILITY REVIEW")
print("=" * 130)


print("\n[1] Frozen pathway libraries")
print("-" * 130)

display(
    gene_set_manifest
)


print("\n[2] MS3 consensus GSEA")
print("-" * 130)

print(
    "Significant terms:",
    len(
        sig_gsea_ms3
    )
)

if len(
    sig_gsea_ms3
) > 0:

    display(

        sig_gsea_ms3[
            [
                col
                for col in [
                    "Library",
                    "Term",
                    "NES",
                    "P_value",
                    "FDR",
                    "Leading_edge"
                ]
                if col in sig_gsea_ms3.columns
            ]
        ]

        .head(50)

        .round(4)
    )


print("\n[3] MS3 vs MS6 GSEA")
print("-" * 130)

print(
    "Significant terms:",
    len(
        sig_gsea_axis
    )
)

if len(
    sig_gsea_axis
) > 0:

    display(

        sig_gsea_axis[
            [
                col
                for col in [
                    "Library",
                    "Term",
                    "NES",
                    "P_value",
                    "FDR",
                    "Leading_edge"
                ]
                if col in sig_gsea_axis.columns
            ]
        ]

        .head(50)

        .round(4)
    )


print("\n[4] Non-cell-cycle residual GSEA")
print("-" * 130)

print(
    "Residual genes after exclusion:",
    len(
        ms3_residual_noncycle
    )
)

print(
    "Significant residual terms:",
    len(
        sig_gsea_residual
    )
)

if len(
    sig_gsea_residual
) > 0:

    display(

        sig_gsea_residual[
            [
                col
                for col in [
                    "Library",
                    "Term",
                    "NES",
                    "P_value",
                    "FDR",
                    "Leading_edge"
                ]
                if col in sig_gsea_residual.columns
            ]
        ]

        .head(60)

        .round(4)
    )


print("\n[5] Residual top genes")
print("-" * 130)

display(

    ms3_residual_noncycle
    .head(50)
    .round(4)
)


print("\n[6] Candidate vulnerability evidence")
print("-" * 130)

display(
    vulnerability_table
    .round(4)
)


print("\n[7] Step26A deep-program context")
print("-" * 130)

display(
    master_program
    .round(4)
)


print("\n[8] Files")
print("-" * 130)


required_files = [

    STEP26B_OBJECT,

    GENESET_MANIFEST_PATH,

    MS3_RANK_PATH,
    AXIS_RANK_PATH,

    GSEA_MS3_PATH,
    GSEA_AXIS_PATH,

    ORA_ALL_PATH,

    BLACKLIST_PATH,
    RESIDUAL_NONCYCLE_PATH,
    RESIDUAL_RANK_PATH,

    GSEA_RESIDUAL_PATH,
    ORA_RESIDUAL_PATH,

    SIG_GSEA_PATH,

    VULNERABILITY_PATH,
    VULNERABILITY_HEAT_PATH,

    STEP26B_MASTER_PATH,

    SUMMARY_PATH,
    MANIFEST_PATH
]


for path in required_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 130)

print(
    "STEP26B STATUS = PATHWAY AND VULNERABILITY EVIDENCE COMPLETE"
)

print(
    "MS3 STATE DEFINITION MODIFIED = NO"
)

print(
    "MS3 CORE50 MODIFIED = NO"
)

print(
    "THERAPEUTIC EFFICACY CLAIMED = NO"
)

print(
    "NEXT = REVIEW RESULTS AND DECIDE WHETHER STEP26 CAN BE FROZEN"
)

print("=" * 130)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP26 FINAL FREEZE - CELL 45
# Reload all Step26 evidence from disk
# ============================================================

from pathlib import Path

import json
import numpy as np
import pandas as pd
import scanpy as sc


# ============================================================
# Paths
# ============================================================

STEP25_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step25_CrossPatient_Robustness')
)

STEP26_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step26_MS3_Biology')
)

DATA_DIR = STEP26_ROOT / "Data"
RESULTS_DIR = STEP26_ROOT / "Results"
FIGURES_DIR = STEP26_ROOT / "Figures"
LOGS_DIR = STEP26_ROOT / "Logs"
REFERENCE_DIR = STEP26_ROOT / "Reference"


# ============================================================
# Inputs
# ============================================================

STEP26B_CHECKPOINT = (
    DATA_DIR
    / "GSE149614_STEP26B_PATHWAY_VULNERABILITY_CHECKPOINT.h5ad"
)

PROGRAM_EVIDENCE_PATH = (
    RESULTS_DIR
    / "Step26A_Master_MS3_MolecularProgram_Evidence.csv"
)

GSEA_MASTER_PATH = (
    RESULTS_DIR
    / "Step26B_GSEA_Significant_Master.csv"
)

VULNERABILITY_INPUT = (
    RESULTS_DIR
    / "Step26B_Candidate_Vulnerability_Evidence.csv"
)

RESIDUAL_GENE_INPUT = (
    RESULTS_DIR
    / "Step26B_MS3_Residual_NonCellCycle_Genes.csv"
)

GENESET_MANIFEST_INPUT = (
    REFERENCE_DIR
    / "Step26B_GeneSet_Library_Manifest.csv"
)

CORE50_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core50.csv"
)

CORE100_INPUT = (
    STEP25_ROOT
    / "Reference"
    / "Step25_FROZEN_MS3_Conserved_Core100.csv"
)


required_inputs = [
    STEP26B_CHECKPOINT,
    PROGRAM_EVIDENCE_PATH,
    GSEA_MASTER_PATH,
    VULNERABILITY_INPUT,
    RESIDUAL_GENE_INPUT,
    GENESET_MANIFEST_INPUT,
    CORE50_INPUT,
    CORE100_INPUT
]


for path in required_inputs:
    assert path.exists(), f"Missing required Step26 input:\n{path}"


# ============================================================
# Load
# ============================================================

adata26 = sc.read_h5ad(
    STEP26B_CHECKPOINT
)

program_evidence = pd.read_csv(
    PROGRAM_EVIDENCE_PATH
)

gsea_master = pd.read_csv(
    GSEA_MASTER_PATH
)

vulnerability = pd.read_csv(
    VULNERABILITY_INPUT
)

residual_genes = pd.read_csv(
    RESIDUAL_GENE_INPUT,
    index_col=0
)

gene_set_manifest = pd.read_csv(
    GENESET_MANIFEST_INPUT
)

core50_table = pd.read_csv(
    CORE50_INPUT,
    index_col=0
)

core100_table = pd.read_csv(
    CORE100_INPUT,
    index_col=0
)


# Remove accidental CSV index columns
for df in [
    program_evidence,
    gsea_master,
    vulnerability,
    gene_set_manifest
]:
    bad_cols = [
        col
        for col in df.columns
        if str(col).startswith("Unnamed:")
    ]

    if len(bad_cols) > 0:
        df.drop(
            columns=bad_cols,
            inplace=True
        )


def extract_gene_list(df):

    for col in [
        "Gene",
        "gene",
        "names"
    ]:
        if col in df.columns:
            return (
                df[col]
                .astype(str)
                .tolist()
            )

    return (
        df.index
        .astype(str)
        .tolist()
    )


MS3_CORE50 = extract_gene_list(
    core50_table
)

MS3_CORE100 = extract_gene_list(
    core100_table
)


print("=" * 120)
print("STEP26 FINAL FREEZE - INPUT AUDIT")
print("=" * 120)

print(
    "Checkpoint:",
    adata26.shape
)

print(
    "Program evidence:",
    program_evidence.shape
)

print(
    "Significant GSEA rows:",
    len(gsea_master)
)

print(
    "Candidate vulnerability genes:",
    len(vulnerability)
)

print(
    "Residual non-cycle genes:",
    len(residual_genes)
)

print(
    "Core50:",
    len(MS3_CORE50)
)

print(
    "Core100:",
    len(MS3_CORE100)
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 46
# Formal biological evidence checks
# ============================================================

# ============================================================
# 1. Primary MS3 core programs
# ============================================================

PRIMARY_POSITIVE_PROGRAMS = [
    "Mitotic_Entry",
    "Chromosomal_Instability",
    "Spindle_Chromosome_Segregation",
    "Cytokinesis"
]


SUPPORTIVE_POSITIVE_PROGRAMS = [
    "Fanconi_DNA_Repair",
    "Nucleotide_Synthesis",
    "Replication_Stress",
    "Homologous_Recombination"
]


OPPOSING_PROGRAMS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism"
]


def get_program_direction(
    program_name
):

    row = program_evidence[
        program_evidence[
            "Program"
        ]
        ==
        program_name
    ]

    assert len(row) == 1, (
        f"Program missing or duplicated: {program_name}"
    )

    return str(
        row.iloc[0][
            "Direction_consensus"
        ]
    )


primary_program_pass = all(

    get_program_direction(
        x
    )
    ==
    "Positive_all"

    for x in PRIMARY_POSITIVE_PROGRAMS
)


supportive_program_pass = all(

    get_program_direction(
        x
    )
    ==
    "Positive_all"

    for x in SUPPORTIVE_POSITIVE_PROGRAMS
)


opposing_program_pass = all(

    get_program_direction(
        x
    )
    ==
    "Negative_all"

    for x in OPPOSING_PROGRAMS
)


# ============================================================
# 2. Residual GSEA after cycle-gene exclusion
# ============================================================

residual_gsea = gsea_master[
    gsea_master[
        "Analysis"
    ]
    .astype(str)
    ==
    "MS3_NonCellCycleResidual"
].copy()


residual_gsea[
    "Term_lower"
] = (
    residual_gsea[
        "Term"
    ]
    .astype(str)
    .str.lower()
)


def residual_term_present(
    keywords
):

    mask = np.ones(
        len(residual_gsea),
        dtype=bool
    )

    for keyword in keywords:

        mask &= (
            residual_gsea[
                "Term_lower"
            ]
            .str.contains(
                keyword.lower(),
                regex=False,
                na=False
            )
        )

    return bool(
        mask.any()
    )


residual_dna_repair_pass = bool(

    residual_term_present(
        ["dna repair"]
    )

    or

    residual_term_present(
        ["double-strand break repair"]
    )
)


residual_replication_fork_pass = bool(

    residual_term_present(
        ["replication fork"]
    )

    or

    residual_term_present(
        ["dna replication"]
    )
)


residual_rna_processing_pass = bool(

    residual_term_present(
        ["rna splicing"]
    )

    or

    residual_term_present(
        ["mrna processing"]
    )
)


residual_chromatin_pass = bool(

    residual_term_present(
        ["chromatin"]
    )

    or

    residual_term_present(
        ["nucleosome organization"]
    )
)


# ============================================================
# 3. Candidate vulnerability support
# ============================================================

PRIMARY_MITOTIC_CANDIDATES = [
    "PLK1",
    "TOP2A",
    "NEK2",
    "BUB1",
    "AURKB",
    "CENPE",
    "TTK",
    "BIRC5",
    "KIF2C",
    "BUB1B",
    "AURKA",
    "CDK1",
    "KIF11"
]


SECONDARY_REPLICATION_CANDIDATES = [
    "RRM2",
    "FANCD2",
    "UBE2T",
    "CHEK1"
]


def candidate_evidence_count(
    gene
):

    row = vulnerability[
        vulnerability[
            "Gene"
        ]
        .astype(str)
        ==
        gene
    ]

    assert len(row) == 1, (
        f"Candidate missing: {gene}"
    )

    return int(
        row.iloc[0][
            "Evidence_count"
        ]
    )


primary_candidate_pass = all(

    candidate_evidence_count(
        gene
    )
    >=
    5

    for gene in PRIMARY_MITOTIC_CANDIDATES
)


secondary_candidate_pass = all(

    candidate_evidence_count(
        gene
    )
    >=
    4

    for gene in SECONDARY_REPLICATION_CANDIDATES
)


# ============================================================
# Final lock
# ============================================================

STEP26_FREEZE_PASS = bool(

    primary_program_pass
    and
    supportive_program_pass
    and
    opposing_program_pass
    and
    residual_dna_repair_pass
    and
    residual_replication_fork_pass
    and
    residual_rna_processing_pass
    and
    residual_chromatin_pass
    and
    primary_candidate_pass
    and
    secondary_candidate_pass
)


print("=" * 120)
print("STEP26 FREEZE EVIDENCE CHECK")
print("=" * 120)

print(
    "Primary mitotic programs conserved:",
    primary_program_pass
)

print(
    "Replication/repair support conserved:",
    supportive_program_pass
)

print(
    "Hepatic/metabolic programs consistently opposing:",
    opposing_program_pass
)

print(
    "Residual DNA-repair enrichment:",
    residual_dna_repair_pass
)

print(
    "Residual replication-fork / DNA-replication enrichment:",
    residual_replication_fork_pass
)

print(
    "Residual RNA-processing enrichment:",
    residual_rna_processing_pass
)

print(
    "Residual chromatin enrichment:",
    residual_chromatin_pass
)

print(
    "Primary mitotic candidate panel supported:",
    primary_candidate_pass
)

print(
    "Secondary replication-stress panel supported:",
    secondary_candidate_pass
)

print("-" * 120)

print(
    "STEP26 FREEZE PASS:",
    STEP26_FREEZE_PASS
)


assert STEP26_FREEZE_PASS, (
    "Step26 freeze criteria are not jointly satisfied. "
    "Review the failed evidence before freezing."
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 47
# Freeze mechanistic candidate panels
# ============================================================

# ============================================================
# Tier 1
# Core proliferative / mitotic identity
# Strongest state-associated candidates
# ============================================================

TIER1_MITOTIC_CORE = [
    "PLK1",
    "TOP2A",
    "NEK2",
    "BUB1",
    "AURKB",
    "CENPE",
    "TTK",
    "BIRC5",
    "KIF2C",
    "BUB1B",
    "AURKA",
    "CDK1",
    "KIF11"
]


# ============================================================
# Tier 2
# Replication-stress / repair candidates
# More mechanistically interesting beyond state identity
# ============================================================

TIER2_REPLICATION_REPAIR = [
    "RRM2",
    "FANCD2",
    "UBE2T",
    "CHEK1"
]


# ============================================================
# Tier 3
# Supportive / context-dependent
# Keep for sensitivity analyses, NOT primary claims
# ============================================================

TIER3_CONTEXTUAL = [
    "WEE1",
    "BRCA1"
]


# ============================================================
# Explicit non-priority candidates
# Current data do not support primary MS3-specific claims
# ============================================================

NOT_PRIMARY_PRIORITY = [
    "ATR",
    "RAD51",
    "CLSPN",
    "MCL1",
    "BCL2L1"
]


frozen_candidate_rows = []


def append_candidate_group(
    genes,
    tier,
    role,
    interpretation
):

    for gene in genes:

        row = vulnerability[
            vulnerability[
                "Gene"
            ]
            .astype(str)
            ==
            gene
        ]


        if len(row) == 1:

            evidence_count = int(
                row.iloc[0][
                    "Evidence_count"
                ]
            )

            mean_delta = float(
                row.iloc[0][
                    "MS3_Mean_delta"
                ]
            )

            axis_delta = float(
                row.iloc[0][
                    "MS3_vs_MS6_Mean_delta"
                ]
            )

            axis_direction = str(
                row.iloc[0][
                    "MS3_vs_MS6_Direction"
                ]
            )

            min_detection = float(
                row.iloc[0][
                    "Min_detection"
                ]
            )

        else:

            evidence_count = np.nan
            mean_delta = np.nan
            axis_delta = np.nan
            axis_direction = "Not_available"
            min_detection = np.nan


        frozen_candidate_rows.append({

            "Gene":
                gene,

            "Priority_tier":
                tier,

            "Future_role":
                role,

            "Interpretation":
                interpretation,

            "Evidence_count":
                evidence_count,

            "MS3_mean_delta":
                mean_delta,

            "MS3_vs_MS6_mean_delta":
                axis_delta,

            "MS3_vs_MS6_direction":
                axis_direction,

            "Minimum_MS3_detection":
                min_detection
        })


append_candidate_group(

    TIER1_MITOTIC_CORE,

    "Tier1",

    "Core_state_and_virtual_perturbation",

    (
        "Strong cross-patient MS3 identity / mitotic machinery. "
        "Prioritized for GRN and virtual perturbation, but expression "
        "support does not itself establish therapeutic efficacy."
    )
)


append_candidate_group(

    TIER2_REPLICATION_REPAIR,

    "Tier2",

    "Mechanistic_dependency_candidate",

    (
        "Replication-stress, nucleotide-supply or DNA-repair candidate "
        "supported beyond the dominant mitotic identity."
    )
)


append_candidate_group(

    TIER3_CONTEXTUAL,

    "Tier3",

    "Sensitivity_only",

    (
        "Biologically plausible but current cross-patient or "
        "MS3-versus-MS6 evidence is not fully concordant."
    )
)


append_candidate_group(

    NOT_PRIMARY_PRIORITY,

    "Not_primary",

    "Do_not_prioritize_without_new_evidence",

    (
        "Current Step26 evidence is insufficient or directionally "
        "inconsistent for a primary MS3-specific claim."
    )
)


frozen_candidate_table = pd.DataFrame(
    frozen_candidate_rows
)


tier_order = {
    "Tier1": 1,
    "Tier2": 2,
    "Tier3": 3,
    "Not_primary": 4
}


frozen_candidate_table[
    "_order"
] = (

    frozen_candidate_table[
        "Priority_tier"
    ]
    .map(
        tier_order
    )
)


frozen_candidate_table = (

    frozen_candidate_table

    .sort_values(
        [
            "_order",
            "Evidence_count",
            "MS3_mean_delta"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )

    .drop(
        columns="_order"
    )

    .reset_index(
        drop=True
    )
)


FROZEN_CANDIDATE_PATH = (
    REFERENCE_DIR
    / "Step26_FROZEN_MS3_Mechanistic_Candidates.csv"
)


frozen_candidate_table.to_csv(

    FROZEN_CANDIDATE_PATH,

    index=False,
    encoding="utf-8-sig"
)


# Plain gene lists
for name, genes in {

    "Tier1_MitoticCore":
        TIER1_MITOTIC_CORE,

    "Tier2_ReplicationRepair":
        TIER2_REPLICATION_REPAIR,

    "Tier3_Contextual":
        TIER3_CONTEXTUAL

}.items():

    path = (
        REFERENCE_DIR
        / f"Step26_FROZEN_{name}.txt"
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        for gene in genes:

            f.write(
                gene
                +
                "\n"
            )


display(
    frozen_candidate_table
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 48
# Freeze biological framework
# ============================================================

step26_biology_framework = pd.DataFrame({

    "Biological_layer": [

        "Primary_state_identity",

        "Primary_state_identity",

        "Primary_state_identity",

        "Primary_state_identity",

        "Supporting_dependency",

        "Supporting_dependency",

        "Supporting_dependency",

        "Supporting_dependency",

        "Residual_biology",

        "Residual_biology",

        "Residual_biology",

        "Residual_biology",

        "Opposing_identity",

        "Opposing_identity"
    ],

    "Program": [

        "Mitotic entry",

        "Chromosomal instability-associated transcription",

        "Spindle / chromosome segregation",

        "Cytokinesis",

        "Replication stress",

        "Fanconi DNA repair",

        "Homologous recombination",

        "Nucleotide synthesis",

        "RNA processing / splicing",

        "Ribosome biogenesis",

        "Replication-fork / DNA maintenance",

        "Chromatin organization / remodeling",

        "Hepatic differentiation",

        "Bile-acid metabolism"
    ],

    "Direction_in_MS3": [

        "Higher",
        "Higher",
        "Higher",
        "Higher",

        "Higher",
        "Higher",
        "Higher",
        "Higher",

        "Higher",
        "Higher",
        "Higher",
        "Higher",

        "Lower",
        "Lower"
    ],

    "Evidence_status": [

        "Cross-patient conserved",
        "Cross-patient conserved",
        "Cross-patient conserved",
        "Cross-patient conserved",

        "Cross-patient supportive",
        "Cross-patient supportive",
        "Cross-patient supportive",
        "Cross-patient supportive",

        "Residual GSEA supported",
        "Residual GSEA supported",
        "Residual GSEA supported",
        "Residual GSEA supported",

        "Cross-patient conserved",
        "Cross-patient conserved"
    ],

    "Interpretation_limit": [

        (
            "Transcriptional state feature; "
            "not direct measurement of proliferation rate."
        ),

        (
            "CIN-associated transcriptional program; "
            "not direct DNA-level CIN measurement."
        ),

        "Transcriptional program",

        "Transcriptional program",

        (
            "Candidate dependency; "
            "functional requirement not yet experimentally proven."
        ),

        (
            "Candidate dependency; "
            "functional requirement not yet experimentally proven."
        ),

        (
            "Candidate dependency; "
            "functional requirement not yet experimentally proven."
        ),

        (
            "Candidate dependency; "
            "functional requirement not yet experimentally proven."
        ),

        (
            "Residual enrichment after operational cycle-gene exclusion; "
            "not fully independent of proliferative biology."
        ),

        (
            "Residual enrichment after operational cycle-gene exclusion."
        ),

        (
            "Residual enrichment after operational cycle-gene exclusion."
        ),

        (
            "Residual enrichment after operational cycle-gene exclusion."
        ),

        (
            "Represents relative loss of mature hepatic identity."
        ),

        (
            "Represents relative loss of mature hepatic metabolic identity."
        )
    ]
})


BIOLOGY_FRAMEWORK_PATH = (
    RESULTS_DIR
    / "Step26_FINAL_Biological_Framework.csv"
)


step26_biology_framework.to_csv(

    BIOLOGY_FRAMEWORK_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    step26_biology_framework
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 49
# Create FINAL FROZEN Step26 object
# ============================================================

adata_final = (
    adata26.copy()
)


adata_final.uns[
    "Step26_final_biology"
] = {

    "Status":
        "FROZEN",

    "Primary_state":
        "MS3",

    "Primary_state_name":
        "Cycling-G2M",

    "Primary_reference_state":
        "MS6",

    "Primary_reference_name":
        "Hepatic-Differentiated",

    "Primary_axis":
        "MS3_vs_MS6",

    "Frozen_MS3_signature":
        "MS3 Conserved Core50",

    "Core50_modified":
        False,

    "State_labels_modified":
        False,

    "Primary_biology":
        [

            "Mitotic entry",

            "Chromosome segregation / spindle checkpoint",

            "Cytokinesis",

            "Chromosomal-instability-associated transcription"
        ],

    "Supporting_biology":
        [

            "Replication stress",

            "Fanconi DNA repair",

            "Homologous recombination",

            "Nucleotide synthesis"
        ],

    "Residual_biology_after_operational_cycle_exclusion":
        [

            "RNA processing and splicing",

            "Ribosome biogenesis",

            "DNA repair",

            "Replication-fork maintenance",

            "Chromatin organization and remodeling",

            "Nucleocytoplasmic transport"
        ],

    "Opposing_biology":
        [

            "Hepatic differentiation",

            "Bile-acid metabolism"
        ],

    "Tier1_mitotic_core":
        TIER1_MITOTIC_CORE,

    "Tier2_replication_repair":
        TIER2_REPLICATION_REPAIR,

    "Tier3_contextual":
        TIER3_CONTEXTUAL,

    "Not_primary_priority":
        NOT_PRIMARY_PRIORITY,

    "Important_interpretation_limits":
        [

            (
                "Chromosomal instability refers to an "
                "associated transcriptional program, not direct "
                "DNA-level measurement of CIN."
            ),

            (
                "Residual analysis uses operational gene exclusion "
                "and does not establish complete independence "
                "from cell-cycle biology."
            ),

            (
                "Candidate vulnerabilities are hypotheses for "
                "future perturbation analyses and do not constitute "
                "therapeutic efficacy evidence."
            )
        ],

    "Future_use":
        {

            "Step27":
                "Trajectory / state-transition analysis",

            "Step29":
                "State-specific GRN analysis",

            "Step33":
                "Virtual gene perturbation",

            "Step34":
                "Virtual state-transition intervention",

            "Step35":
                "Drug-reversal analysis"
        }
}


STEP26_FINAL_OBJECT = (
    DATA_DIR
    / "GSE149614_STEP26_FINAL_FROZEN_MS3_BIOLOGY.h5ad"
)


adata_final.write_h5ad(

    STEP26_FINAL_OBJECT,

    compression="gzip"
)


print("=" * 110)
print("STEP26 FINAL OBJECT SAVED")
print("=" * 110)

print(
    STEP26_FINAL_OBJECT
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 50
# Final decision record
# ============================================================

step26_final_decision = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step26_MS3_Deep_Biological_Characterization",

    "Status":
        "FROZEN",

    "Primary_state":
        {

            "ID":
                "MS3",

            "Name":
                "Cycling-G2M",

            "Bulk_risk_role":
                "Primary bulk-risk-concordant",

            "Cross_patient_status":
                "Internally portable across HCC08/HCC09/HCC10"
        },

    "Primary_reference":
        {

            "ID":
                "MS6",

            "Name":
                "Hepatic-Differentiated",

            "Bulk_risk_role":
                "Primary bulk-risk-opposing"
        },

    "Frozen_primary_biological_model":
        (

            "MS3 represents a cross-patient proliferative malignant "
            "state characterized primarily by mitotic entry, spindle "
            "and chromosome-segregation machinery, cytokinesis and "
            "chromosomal-instability-associated transcription. "
            "The state additionally shows replication-stress, "
            "DNA-repair and nucleotide-synthesis requirements, "
            "with residual enrichment of RNA processing, ribosome "
            "biogenesis, DNA maintenance and chromatin regulation "
            "after operational exclusion of dominant cell-cycle genes. "
            "MS3 is opposed by loss of hepatic differentiation and "
            "bile-acid metabolic identity."
        ),

    "Primary_state_identity_programs":
        PRIMARY_POSITIVE_PROGRAMS,

    "Supporting_dependency_programs":
        SUPPORTIVE_POSITIVE_PROGRAMS,

    "Opposing_programs":
        OPPOSING_PROGRAMS,

    "Frozen_candidate_panels":
        {

            "Tier1_MitoticCore":
                TIER1_MITOTIC_CORE,

            "Tier2_ReplicationRepair":
                TIER2_REPLICATION_REPAIR,

            "Tier3_Contextual":
                TIER3_CONTEXTUAL,

            "NotPrimaryPriority":
                NOT_PRIMARY_PRIORITY
        },

    "Residual_analysis":
        {

            "Status":
                "Supported",

            "Interpretation":
                (
                    "Operational removal of frozen Core100 plus "
                    "Hallmark G2M/E2F and curated mitotic genes "
                    "did not eliminate DNA-repair, replication-fork, "
                    "RNA-processing and chromatin-related enrichment."
                ),

            "Important_limit":
                (
                    "Residual enrichment must not be interpreted as "
                    "fully independent of proliferative biology."
                )
        },

    "Therapeutic_efficacy_claimed":
        False,

    "Functional_dependency_proven":
        False,

    "Chromosomal_instability_directly_measured":
        False,

    "State_labels_modified":
        False,

    "Core50_modified":
        False,

    "Step26_freeze_pass":
        STEP26_FREEZE_PASS,

    "Next_step":
        (

            "Step27 trajectory analysis will examine whether the "
            "frozen malignant states occupy reproducible transition "
            "relationships, with particular attention to the "
            "MS6 differentiated/metabolic to MS3 proliferative axis. "
            "Trajectory directionality will not be assumed in advance."
        )
}


FINAL_DECISION_PATH = (
    LOGS_DIR
    / "Step26_FINAL_Decision.json"
)


with open(

    FINAL_DECISION_PATH,

    "w",
    encoding="utf-8"

) as f:

    json.dump(

        step26_final_decision,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    FINAL_DECISION_PATH
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 51
# Publication-oriented final summary
# ============================================================

step26_publication_summary = pd.DataFrame({

    "Item": [

        "Primary malignant state",
        "Primary reference state",

        "Primary biological identity",

        "Primary proliferative architecture",

        "Supporting replication biology",

        "Residual biology after cycle exclusion",

        "Opposing biological identity",

        "Primary candidate tier",

        "Mechanistic dependency candidate tier",

        "Context-dependent candidates",

        "Non-priority candidates",

        "Chromosomal instability interpretation",

        "Therapeutic efficacy",

        "State labels modified",

        "Core50 modified",

        "Next analysis"
    ],

    "Result": [

        "MS3 Cycling-G2M",

        "MS6 Hepatic-Differentiated",

        (
            "Cross-patient proliferative / mitotic "
            "malignant-cell state"
        ),

        (
            "Mitotic entry | spindle checkpoint | "
            "chromosome segregation | cytokinesis"
        ),

        (
            "Replication stress | Fanconi DNA repair | "
            "homologous recombination | nucleotide synthesis"
        ),

        (
            "RNA processing/splicing | ribosome biogenesis | "
            "DNA repair/replication-fork maintenance | "
            "chromatin organization"
        ),

        (
            "Reduced hepatic differentiation and "
            "bile-acid metabolism"
        ),

        "|".join(
            TIER1_MITOTIC_CORE
        ),

        "|".join(
            TIER2_REPLICATION_REPAIR
        ),

        "|".join(
            TIER3_CONTEXTUAL
        ),

        "|".join(
            NOT_PRIMARY_PRIORITY
        ),

        (
            "CIN-associated transcriptional program; "
            "not direct DNA-level CIN measurement"
        ),

        "Not established",

        "No",

        "No",

        "Step27 trajectory / malignant-state transition analysis"
    ]
})


PUBLICATION_SUMMARY_PATH = (
    RESULTS_DIR
    / "Step26_FINAL_Publication_Summary.csv"
)


step26_publication_summary.to_csv(

    PUBLICATION_SUMMARY_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    step26_publication_summary
)

In [ ]:
# ============================================================
# STEP26 FINAL FREEZE - CELL 52
# Final manifest and freeze review
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),

    ("Results", RESULTS_DIR),

    ("Figures", FIGURES_DIR),

    ("Logs", LOGS_DIR),

    ("Reference", REFERENCE_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.rglob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Relative_path":
                    str(
                        path.relative_to(
                            STEP26_ROOT
                        )
                    ),

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


FINAL_MANIFEST_PATH = (
    RESULTS_DIR
    / "Step26_FINAL_Output_Manifest.csv"
)


manifest_df.to_csv(

    FINAL_MANIFEST_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 130)
print("STEP26 FINAL FREEZE REVIEW")
print("=" * 130)


print("\n[1] Primary biological model")
print("-" * 130)

print(
    "Primary state: MS3 Cycling-G2M"
)

print(
    "Reference state: MS6 Hepatic-Differentiated"
)

print(
    "Core identity:"
)

for x in PRIMARY_POSITIVE_PROGRAMS:
    print("  -", x)


print(
    "\nSupporting replication / repair programs:"
)

for x in SUPPORTIVE_POSITIVE_PROGRAMS:
    print("  -", x)


print(
    "\nOpposing programs:"
)

for x in OPPOSING_PROGRAMS:
    print("  -", x)


print("\n[2] Frozen candidate hierarchy")
print("-" * 130)

print(
    "Tier 1:",
    ", ".join(
        TIER1_MITOTIC_CORE
    )
)

print(
    "Tier 2:",
    ", ".join(
        TIER2_REPLICATION_REPAIR
    )
)

print(
    "Tier 3:",
    ", ".join(
        TIER3_CONTEXTUAL
    )
)

print(
    "Not primary:",
    ", ".join(
        NOT_PRIMARY_PRIORITY
    )
)


print("\n[3] Residual-biology evidence")
print("-" * 130)

print(
    "DNA repair:",
    residual_dna_repair_pass
)

print(
    "Replication fork / DNA replication:",
    residual_replication_fork_pass
)

print(
    "RNA processing:",
    residual_rna_processing_pass
)

print(
    "Chromatin regulation:",
    residual_chromatin_pass
)


print("\n[4] Interpretation limits")
print("-" * 130)

print(
    "CIN directly measured = NO"
)

print(
    "Functional dependency proven = NO"
)

print(
    "Therapeutic efficacy established = NO"
)

print(
    "Residual biology fully independent of cell cycle = NO"
)


print("\n[5] Frozen files")
print("-" * 130)


required_final_files = [

    STEP26_FINAL_OBJECT,

    FROZEN_CANDIDATE_PATH,

    BIOLOGY_FRAMEWORK_PATH,

    FINAL_DECISION_PATH,

    PUBLICATION_SUMMARY_PATH,

    FINAL_MANIFEST_PATH,

    (
        REFERENCE_DIR
        / "Step26_FROZEN_Tier1_MitoticCore.txt"
    ),

    (
        REFERENCE_DIR
        / "Step26_FROZEN_Tier2_ReplicationRepair.txt"
    ),

    (
        REFERENCE_DIR
        / "Step26_FROZEN_Tier3_Contextual.txt"
    )
]


for path in required_final_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 130)

print(
    "STEP26 STATUS = FROZEN"
)

print(
    "PRIMARY BIOLOGY = MITOTIC / CHROMOSOME-SEGREGATION STATE"
)

print(
    "SECONDARY BIOLOGY = REPLICATION-STRESS / DNA-REPAIR DEPENDENCY"
)

print(
    "OPPOSING BIOLOGY = HEPATIC DIFFERENTIATION / METABOLISM"
)

print(
    "PRIMARY VULNERABILITY PANEL = FROZEN"
)

print(
    "THERAPEUTIC EFFICACY CLAIMED = NO"
)

print(
    "NEXT = STEP27 MALIGNANT-STATE TRAJECTORY ANALYSIS"
)

print("=" * 130)